# Heart Failure Project
## Statistical learning for healthcare data project

**Authors:** Mattia Alborghetti, Alessandro Ambrogi, Francesco Colombo.

Collaborative course project, Statistical Learning for Healthcare Data, Politecnico di Milano.


This project focuses on predicting hospital readmission within 6 months for patients affected by heart failure using clinical and laboratory information collected during hospitalization.

The dataset originates from the Heart Failure Zigong database and includes demographic variables, laboratory measurements, comorbidities, medications and echocardiographic information.

The main goal is to build a clinically interpretable machine learning pipeline capable of identifying patients at high risk of readmission after discharge.

## Public portfolio edition
The original analysis is preserved; outputs and student identifiers have been removed.
Obtain the restricted Zigong data directly from PhysioNet under its data-use agreement; see README.md and DATA_ACCESS.md. No patient data are distributed here.
Set HF_DATA_DIR to your private data directory containing dat.csv and dat_md.csv.
Historical model metrics and important validation limitations are documented in RESULTS.md. They have not been independently reproduced for this edition.


Importing all the required libraries:

In [ ]:
# ==============================
# Core libraries for data handling and visualization
# ==============================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

import seaborn as sns
import warnings

# ==============================
# Statistical tests
# ==============================

from scipy.stats import mannwhitneyu, kruskal
from scipy.stats import chi2_contingency

# ==============================
# Data splitting and imputation
# ==============================

from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

# ==============================
# Base classes and preprocessing
# ==============================

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ==============================
# Feature selection
# ==============================

from sklearn.feature_selection import RFE
from sklearn.feature_selection import RFECV
from boruta import BorutaPy

# ==============================
# Machine learning models
# ==============================

from sklearn.ensemble import VotingClassifier
from interpret.glassbox import ExplainableBoostingClassifier

from sklearn.ensemble import (
    StackingClassifier,
    RandomForestClassifier,
    GradientBoostingClassifier
)

from xgboost import XGBClassifier, XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# ==============================
# Cross-validation
# ==============================

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import  cross_val_score
from sklearn.linear_model import LogisticRegressionCV

# ==============================
# Model calibration
# ==============================

from sklearn.calibration import CalibratedClassifierCV  
import optuna

# ==============================
# Evaluation metrics
# ==============================

from sklearn.metrics import (
    roc_auc_score, roc_curve, auc, 
    precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report, f1_score, recall_score, precision_score 
)
import shap






## Data Loading

We load the main clininal dataset and the medication dataset.

In [ ]:
# ==============================
# clinical dataset
# ==============================

import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("HF_DATA_DIR", "data"))
path_1 = DATA_DIR / "dat.csv"
if not path_1.is_file():
    raise FileNotFoundError("Place authorized PhysioNet data in data/ or set HF_DATA_DIR; see DATA_ACCESS.md")
df = pd.read_csv(path_1)

# ==============================
# medication dataset
# ==============================

path_2 = DATA_DIR / "dat_md.csv"
md_df = pd.read_csv(path_2)

Fixing data type in order to be easier later on to work with them:

In [ ]:
df = df.astype({

    # =========================
    # CATEGORICAL
    # =========================
    'DestinationDischarge': 'category',
    'admission.ward': 'category',
    'admission.way': 'category',
    'occupation': 'category',
    'discharge.department': 'category',
    'gender': 'category',
    'type.of.heart.failure': 'category',
    'NYHA.cardiac.function.classification': 'category',
    'Killip.grade': 'category',
    'type.II.respiratory.failure': 'category',
    'consciousness': 'category',
    'respiratory.support.': 'category',
    'oxygen.inhalation': 'category',
    'outcome.during.hospitalization': 'category',
    'ageCat': 'category',
    'GCS': 'category',
    'eye.opening': 'category',
    'verbal.response': 'category',
    'movement': 'category',

    # =========================
    # BOOLEAN
    # =========================
    'myocardial.infarction': 'bool',
    'congestive.heart.failure': 'bool',
    'peripheral.vascular.disease': 'bool',
    'cerebrovascular.disease': 'bool',
    'dementia': 'bool',
    'Chronic.obstructive.pulmonary.disease': 'bool',
    'connective.tissue.disease': 'bool',
    'peptic.ulcer.disease': 'bool',
    'diabetes': 'bool',
    'moderate.to.severe.chronic.kidney.disease': 'bool',
    'hemiplegia': 'bool',
    'leukemia': 'bool',
    'malignant.lymphoma': 'bool',
    'solid.tumor': 'bool',
    'liver.disease': 'bool',
    'AIDS': 'bool',
    'acute.renal.failure': 'bool',

    # =========================
    # OUTCOME-RELATED
    # =========================

    'death.within.28.days': 'bool',
    're.admission.within.28.days': 'bool',
    'death.within.3.months': 'bool',
    're.admission.within.3.months': 'bool',
    'death.within.6.months': 'bool',
    're.admission.within.6.months': 'bool',
    'return.to.emergency.department.within.6.months': 'bool'

})

## Dataset Overview


**Key characteristics:**

- Outcome prevalence: Not readmitted: 61.5% / Readmitted: 38.5%
- Gender distribution: Female: 57.9% / Male: 42.1%
- Age distribution: (21,29]: 0.2% / (31,39]: 0.6% / (41,49]: 2.8% / (51,59]: 5.3% / (61,69]: 18.3% / (71,79]: 35.6% / (81,89]: 32.2% / (89,110]: 5.0%
- Type of heart failure: Left: 23.8% / Both: 73.7% / Right: 2.5%


In [ ]:
# =========================================================
# DATASET OVERVIEW DASHBOARD
# =========================================================


# ---------------------------------------------------------
# Summary Table
# ---------------------------------------------------------

summary = pd.DataFrame({

    "Metric": [
        "Patients",
        "Variables",
        "Readmission Rate (%)",
        "Female (%)",
        "Male (%)"
    ],

    "Value": [
        df.shape[0],
        df.shape[1],
        round(
            df['re.admission.within.6.months']
            .mean() * 100,
            2
        ),
        round(
            (
                df['gender']
                .eq('Female')
                .mean()
            ) * 100,
            2
        ),
        round(
            (
                df['gender']
                .eq('Male')
                .mean()
            ) * 100,
            2
        )
    ]
})

# ---------------------------------------------------------
# Dashboard
# ---------------------------------------------------------

fig, axes = plt.subplots(
    2, 2,
    figsize=(14,10)
)

# =========================================================
# Readmission Distribution
# =========================================================

target_pct = (
    df['re.admission.within.6.months']
    .value_counts(normalize=True)
    .mul(100)
)

axes[0,0].bar(
    target_pct.index.astype(str),
    target_pct.values
)

axes[0,0].set_title(
    "Readmission within 6 Months"
)

axes[0,0].set_ylabel("% Patients")

for i, v in enumerate(target_pct.values):

    axes[0,0].text(
        i,
        v + 1,
        f"{v:.1f}%",
        ha='center'
    )

# =========================================================
# Gender Distribution
# =========================================================

gender_pct = (
    df['gender']
    .value_counts(normalize=True)
    .mul(100)
)

axes[0,1].bar(
    gender_pct.index,
    gender_pct.values
)

axes[0,1].set_title(
    "Gender Distribution"
)

axes[0,1].set_ylabel("% Patients")

for i, v in enumerate(gender_pct.values):

    axes[0,1].text(
        i,
        v + 1,
        f"{v:.1f}%",
        ha='center'
    )



# =========================================================
# Age Distribution
# =========================================================

age_pct = (
    df['ageCat']
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

axes[1,0].bar(
    age_pct.index.astype(str),
    age_pct.values
)

axes[1,0].set_title(
    "Age Distribution"
)

axes[1,0].set_ylabel("% Patients")

axes[1,0].tick_params(
    axis='x',
    rotation=45
)

for i, v in enumerate(age_pct.values):

    axes[1,0].text(
        i,
        v + 1,
        f"{v:.1f}%",
        ha='center'
    )


# =========================================================
# Heart Failure Type
# =========================================================

hf_pct = (
    df['type.of.heart.failure']
    .value_counts(normalize=True)
    .mul(100)
)

axes[1,1].bar(
    hf_pct.index,
    hf_pct.values
)

axes[1,1].set_title(
    "Type of Heart Failure"
)

axes[1,1].set_ylabel("% Patients")

for i, v in enumerate(hf_pct.values):

    axes[1,1].text(
        i,
        v + 1,
        f"{v:.1f}%",
        ha='center'
    )

plt.suptitle(
    "Dataset Overview",
    fontsize=18,
    fontweight='bold'
)

plt.tight_layout()

plt.show()

# =========================================================
# MOST COMMON COMORBIDITIES
# =========================================================

comorb_cols = [

    'diabetes',
    'myocardial.infarction',
    'cerebrovascular.disease',
    'Chronic.obstructive.pulmonary.disease',
    'moderate.to.severe.chronic.kidney.disease',
    'liver.disease',
    'acute.renal.failure',
    'congestive.heart.failure'

]

comorb_pct = (
    df[comorb_cols]
    .mean()
    .mul(100)
    .sort_values()
)

plt.figure(figsize=(8,5))

comorb_pct.plot.barh()

plt.xlabel("% of Patients")
plt.title(
    "Most Common Comorbidities",
    fontweight='bold'
)

plt.tight_layout()
plt.show()

readm_rates = pd.DataFrame({

    "Time Horizon": [
        "28 Days",
        "3 Months",
        "6 Months"
    ],

    "Rate": [

        df['re.admission.within.28.days']
        .mean()*100,

        df['re.admission.within.3.months']
        .mean()*100,

        df['re.admission.within.6.months']
        .mean()*100

    ]

})

plt.figure(figsize=(9,5))

# =====================================================
# READMISSION
# =====================================================

plt.plot(
    readm_rates["Time Horizon"],
    readm_rates["Rate"],
    marker='o',
    linewidth=2,
    markersize=8,
    label='Readmission'
)

# =====================================================
# MORTALITY
# =====================================================

mortality_rates = [
    df['death.within.28.days'].mean()*100,
    df['death.within.3.months'].mean()*100,
    df['death.within.6.months'].mean()*100
]

mortality_counts = [
    df['death.within.28.days'].sum(),
    df['death.within.3.months'].sum(),
    df['death.within.6.months'].sum()
]

plt.plot(
    readm_rates["Time Horizon"],
    mortality_rates,
    marker='s',
    linewidth=2,
    markersize=8,
    label='Mortality'
)

# =====================================================
# READMISSION LABELS
# =====================================================

for x, y, n in zip(
    readm_rates["Time Horizon"],
    readm_rates["Rate"],
    [
        df['re.admission.within.28.days'].sum(),
        df['re.admission.within.3.months'].sum(),
        df['re.admission.within.6.months'].sum()
    ]
):

    plt.annotate(
        f"{y:.1f}%\n(n={n})",
        (x, y),
        xytext=(0, 12),
        textcoords='offset points',
        ha='center'
    )

# =====================================================
# MORTALITY LABELS
# =====================================================

for x, y, n in zip(
    readm_rates["Time Horizon"],
    mortality_rates,
    mortality_counts
):

    plt.annotate(
        f"{y:.1f}%\n(n={n})",
        (x, y),
        xytext=(0, 10),
        textcoords='offset points',
        ha='center',
        fontsize=9
    )

# spazio extra sopra il 38.6%
plt.ylim(
    0,
    max(readm_rates["Rate"]) + 8
)

plt.ylabel("Rate (%)")

plt.title(
    "Readmission and Mortality Across Follow-Up Horizons",
    fontweight='bold'
)

plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

We categorize variables based on their data type.

In [ ]:
continuous_vars = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_vars = df.select_dtypes(include=['category']).columns.tolist()
boolean_vars = df.select_dtypes(include=['bool']).columns.tolist()

print(f"Number of continuous covariates: {len(continuous_vars)}")
print(f"Number of categorical covariates: {len(categorical_vars)}")
print(f"Number of boolean covariates: {len(boolean_vars)}")

Before proceeding with the analysis, we perform a series of data quality checks 
to ensure the reliability and clinical consistency of the dataset.

This step includes:

- Removal of duplicate records
- Identification and correction of logical inconsistencies: 
            
    - If a patient is discharged as *"Died"*, then the clinical outcome must be *"Dead"*
    - Any mismatch indicates a data inconsistency

In [ ]:
# =========================================================
# INITIAL DATA CONSISTENCY CHECKS
# =========================================================

print("=== INITIAL DATA QUALITY CHECK ===\n")

# ---------------------------------------------------------
# 1. DUPLICATE ROWS
# ---------------------------------------------------------

n_duplicates = df.duplicated().sum()

print(f"Duplicate rows found: {n_duplicates}")

if n_duplicates > 0:
    # Remove duplicate entries
    print(f"Removing {n_duplicates} duplicate rows...\n")
    df = df.drop_duplicates()
else:
    print("No duplicate rows found.\n")


# ---------------------------------------------------------
# 2. INCONSISTENT DEATH INFORMATION
# ---------------------------------------------------------

# Case 1: Discharge says "Died" but outcome is not "Dead"
inconsistent_discharge = df[
    (df['DestinationDischarge'] == 'Died') &
    (df['outcome.during.hospitalization'] != 'Dead')
]

# Case 2: Outcome says "Dead" but discharge is not "Died"
inconsistent_outcome = df[
    (df['DestinationDischarge'] != 'Died') &
    (df['outcome.during.hospitalization'] == 'Dead')
]

# Report inconsistencies
print(
    "Inconsistencies: "
    f"{len(inconsistent_discharge)} patients "
    "with discharge='Died' but outcome!='Dead'"
)

print(
    "Inconsistencies: "
    f"{len(inconsistent_outcome)} patients "
    "with outcome='Dead' but discharge!='Died'\n"
)

# Total number of inconsistent records
total_inconsistent = len(inconsistent_discharge) + len(inconsistent_outcome)

print(f"Total inconsistent rows removed: {total_inconsistent}\n")

# Remove inconsistent records from the dataset
df = df.drop(inconsistent_discharge.index)
df = df.drop(inconsistent_outcome.index)


# ---------------------------------------------------------
# FINAL DATASET SHAPE
# ---------------------------------------------------------

print("Final dataset shape after cleaning:")
print(df.shape)

we remove variables that could introduce **data leakage** into the model, including these variables would allow the model to indirectly "see the future".

In [ ]:
# =========================================================
# REMOVAL OF "FUTURE-SEEING" FEATURES
# =========================================================

features_to_remove = [

    'death.within.28.days',
    're.admission.within.28.days',

    'death.within.3.months',
    're.admission.within.3.months',

    'time.of.death..days.from.admission.',
    're.admission.time..days.from.admission.',

    'return.to.emergency.department.within.6.months',
    'time.to.emergency.department.within.6.months'

    'time.of.death..days.from.admission.',
    're.admission.time..days.from.admission.',
    'time.to.emergency.department.within.6.months'
]

# Drop selected features
df = df.drop(
    columns=features_to_remove,
    errors='ignore'
)

The dataset is divided into:

- **Continuous variables** → numerical features
- **Categorical variables** → discrete features with multiple levels
- **Boolean variables** → binary indicators


In [ ]:
# =========================================================
# FEATURE TYPE IDENTIFICATION
# =========================================================

continuous_vars = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_vars = df.select_dtypes(include=['category']).columns.tolist()
boolean_vars = df.select_dtypes(include=['bool']).columns.tolist()

# ---------------------------------------------------------
# SUMMARY OF FEATURE TYPES
# ---------------------------------------------------------

print(f"Number of continuous covariates: {len(continuous_vars)}")
print(f"Number of categorical covariates: {len(categorical_vars)}")
print(f"Number of boolean covariates: {len(boolean_vars)}")

## Exploratory Analysis of Categorical Variables

For each variable, we visualize:

- Absolute frequency of each category
- Presence of class imbalance

In [ ]:
# =========================================================
# GRID CONFIGURATION FOR PLOTTING
# =========================================================

# Number of columns in the grid
n_cols = 5

n_rows = math.ceil(len(categorical_vars) / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(18, 5 * n_rows)
)

axes = axes.flatten()


# =========================================================
# LOOP THROUGH CATEGORICAL VARIABLES
# =========================================================

for i, col in enumerate(categorical_vars):

    ax = axes[i]

    abs_freq = df[col].value_counts(dropna=False)

    sns.countplot(
        data=df,
        x=col,
        order=abs_freq.index,
        ax=ax
    )

    ax.set_title(col, fontsize=10)

    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.tick_params(axis='x', rotation=45)

    # -----------------------------------------------------
    # Annotate bars with counts
    # -----------------------------------------------------
    for p in ax.patches:
        height = p.get_height()

        if height > 0:
            ax.annotate(
                f'{int(height)}',
                (p.get_x() + p.get_width() / 2, height),
                ha='center',
                va='bottom',
                fontsize=7
            )


# =========================================================
# CLEANUP EMPTY SUBPLOTS
# =========================================================

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])


# =========================================================
# FINAL LAYOUT
# =========================================================

plt.tight_layout()
plt.show()

### Encoding of Ordinal Clinical Variables

The following variables are encoded as ordinal based on established clinical scales:

- **NYHA classification** → severity of heart failure (I → IV)
- **Killip grade** → severity of acute cardiac dysfunction (I → IV)
- **Glasgow Coma Scale (GCS)** → level of consciousness
- **GCS components** → neurological response sub-scores
- **Age categories** → naturally ordered demographic groups

In [ ]:
# =========================================================
# DEFINE ORDINAL VARIABLES CORRECTLY
# =========================================================

# ---------------------------------------------------------
# NYHA CLASSIFICATION
# Clinical severity scale:
# I < II < III < IV
# ---------------------------------------------------------

df['NYHA.cardiac.function.classification'] = pd.Categorical(
    df['NYHA.cardiac.function.classification'],
    categories=['I', 'II', 'III', 'IV'],
    ordered=True
)


# ---------------------------------------------------------
# KILLIP GRADE
# Clinical severity scale:
# I < II < III < IV
# ---------------------------------------------------------

df['Killip.grade'] = pd.Categorical(
    df['Killip.grade'],
    categories=['I', 'II', 'III', 'IV'],
    ordered=True
)


# ---------------------------------------------------------
# AGE CATEGORIES
# ---------------------------------------------------------

age_order = sorted(
    df['ageCat']
    .dropna()
    .unique()
)

df['ageCat'] = pd.Categorical(
    df['ageCat'],
    categories=age_order,
    ordered=True
)


# ---------------------------------------------------------
# GLASGOW COMA SCALE (GCS)
# Total score: 3 (worst) → 15 (best)
# ---------------------------------------------------------

df['GCS'] = pd.Categorical(
    df['GCS'],
    categories=list(range(3, 16)),
    ordered=True
)


# ---------------------------------------------------------
# GCS COMPONENTS
# ---------------------------------------------------------

# Eye opening: 1 (none) → 4 (spontaneous)
df['eye.opening'] = pd.Categorical(
    df['eye.opening'],
    categories=[1, 2, 3, 4],
    ordered=True
)

# Verbal response: 1 (none) → 5 (oriented)
df['verbal.response'] = pd.Categorical(
    df['verbal.response'],
    categories=[1, 2, 3, 4, 5],
    ordered=True
)

# Motor response: 1 (none) → 6 (obeys commands)
df['movement'] = pd.Categorical(
    df['movement'],
    categories=[1, 2, 3, 4, 5, 6],
    ordered=True
)


# =========================================================
# CHECK TYPES
# =========================================================

print("\n=== UPDATED ORDINAL VARIABLES ===\n")

print(
    df[
        [
            'NYHA.cardiac.function.classification',
            'Killip.grade',
            'ageCat',
            'GCS',
            'eye.opening',
            'verbal.response',
            'movement'
        ]
    ].dtypes
)

### Analysis of the possible difference between different outcome patient

We analyze the distribution of the target variable:

**`outcome.during.hospitalization`**

This step is essential to understand:

- The proportion of patients in each outcome category
- The presence of class imbalance

In [ ]:
# =========================================================
# OUTCOME DISTRIBUTION ANALYSIS
# =========================================================

# Compute the number of observations for each outcome category
outcome_counts = df['outcome.during.hospitalization'].value_counts(dropna=False)

# Display the distribution of the target variable
print("Number of data points in each outcome category:")
print(outcome_counts)

In this step, we remove patients who died during hospitalization 
(`outcome.during.hospitalization = "Dead"`).

The objective of the analysis is to focus on **post-discharge outcomes** 
(e.g., readmission), which are only defined for patients who survive hospitalization.

- Death during hospitalization is a **competing event**
- Patients who die cannot experience future outcomes (e.g., readmission)
- Removing them ensures a **clinically coherent population**

In [ ]:
# =========================================================
# EXCLUDE DECEASED PATIENTS
# =========================================================

df = df[df['outcome.during.hospitalization'] != 'Dead']

# ---------------------------------------------------------
# CLEAN CATEGORICAL LEVELS
# ---------------------------------------------------------

df['outcome.during.hospitalization'] = (
    df['outcome.during.hospitalization']
    .cat.remove_unused_categories()
)

# ---------------------------------------------------------
# VERIFY UPDATED DISTRIBUTION
# ---------------------------------------------------------

print("Updated outcome categories:")
print(df['outcome.during.hospitalization'].value_counts(dropna=False))

### Univariate Analysis of Continuous variables
 
The goal is to identify features that show statistically significant differences across outcome groups.

We apply the **Kruskal–Wallis test**, a non-parametric method used to compare multiple independent groups.

This choice is motivated by:

- Presence of outliers
- Heterogeneity across patient groups

Unlike ANOVA, Kruskal–Wallis does not assume normality and is more robust for real-world clinical data.

In [ ]:
# =========================================================
# EFFECT SIZE FUNCTION (COHEN'S D)
# =========================================================

def cohens_d(g0, g1):
    """Compute standardized mean difference between two groups."""
    
    n0, n1 = len(g0), len(g1)
    var0, var1 = g0.var(), g1.var()
    
    # Pooled standard deviation
    pooled_std = np.sqrt(
        ((n0 - 1) * var0 + (n1 - 1) * var1) / (n0 + n1 - 2)
    )
    
    # Avoid division by zero
    if pooled_std == 0:
        return 0
    
    return (g1.mean() - g0.mean()) / pooled_std


In [ ]:
# =========================================================
# UNIVARIATE STATISTICAL ANALYSIS
# =========================================================

results_hosp = []
target_col = 'outcome.during.hospitalization'

# Extract unique outcome categories (excluding missing values)
categories = df[target_col].dropna().unique()


# ---------------------------------------------------------
# LOOP OVER CONTINUOUS VARIABLES
# ---------------------------------------------------------

for col in continuous_vars:

    # Split the variable into groups based on outcome categories
    groups = [
        df[df[target_col] == cat][col].dropna()
        for cat in categories
    ]
    
    # Remove groups with very small sample size
    groups = [g for g in groups if len(g) > 10]
    
    # Kruskal-Wallis requires at least 2 groups
    if len(groups) >= 2:
        try:
            stat, p = kruskal(*groups)
            
            results_hosp.append({
                'feature': col,
                'p_value': p
            })
        
        except ValueError as e:
            # Handle edge case: all values identical
            if "All numbers are identical" in str(e):
                continue
            else:
                raise e


# =========================================================
# RESULTS RANKING
# =========================================================

# Convert results to DataFrame and sort by p-value
df_hosp = pd.DataFrame(results_hosp).sort_values('p_value')

# Select top features
top_features_hosp = df_hosp.head(12)

print("\nTop continuous variables associated with outcome:")
print(top_features_hosp)


# =========================================================
# VISUALIZATION: BOXPLOTS
# =========================================================

# Extract top feature names
top_12_hosp = top_features_hosp['feature'].head(12).tolist()

# Create subplot grid (4 rows x 3 columns)
fig, axes = plt.subplots(4, 3, figsize=(14, 20))

fig.suptitle(
    'Distributions of Top Features by Hospitalization Outcome',
    fontsize=16
)

# Plot each feature
for i, feature in enumerate(top_12_hosp):

    ax = axes[i // 3, i % 3]
    
    sns.boxplot(
        data=df,
        x=target_col,
        y=feature,
        ax=ax,
        palette='Pastel1',
        hue=target_col,
        legend=False
    )
    
    ax.set_title(feature)
    ax.set_xlabel("Outcome")
    ax.set_ylabel("Value")
    ax.tick_params(axis='x', rotation=45)


# Adjust layout
plt.tight_layout()
plt.show()

By looking at the plot of the outcome of Alive and DischargeAgainstOrder we can see that even the most dissimilar features are nearly the same.
Hence, we can safely discard this covariate.

In [ ]:
# =========================================================
# REMOVE TARGET VARIABLE FROM FEATURE SET
# =========================================================

features_to_remove = [
    'outcome.during.hospitalization'
]

# Drop the target column from the dataset
df = df.drop(
    columns=features_to_remove,
    errors='ignore'  # Avoid errors if column is already removed
)

#### Analysis for 'death.within.6.months'

We want to assess if patient that dies within 6 months and the one that get rehospitalized have similar features or not.

In [ ]:
# --- Print the counts for both variables ---
print("Counts for 're.admission.within.6.months':")
print(df['re.admission.within.6.months'].value_counts(dropna=False))

print("\nCounts for 'death.within.6.months':")
print(df['death.within.6.months'].value_counts(dropna=False))
print("-" * 50)

The amount of dead patient seems negligible at first, since 46 << 1946, however we still want more evidence, we are going to look if dead patients and the one getting recovered are similar or not. 

(with this idea we are going to look if the patient that are dead should be treated as the ill one getting recovered again or not)

We use the **Mann–Whitney U test**, a non-parametric test for comparing two independent groups.

This choice is appropriate because:

- Clinical variables are often non-normally distributed
- The test is robust to outliers
- It does not assume equal variances

In [ ]:
# =========================================================
# UNIVARIATE COMPARISON: 6-MONTH OUTCOMES
# =========================================================

results_6m = []

for col in continuous_vars:

    # Group 0: patients readmitted within 6 months
    g0 = df[df['re.admission.within.6.months'] == 1][col].dropna()
    
    # Group 1: patients who died within 6 months
    g1 = df[df['death.within.6.months'] == 1][col].dropna()
    
    # Ensure sufficient sample size
    if len(g0) > 10 and len(g1) > 10:
        
        # Mann-Whitney U test (non-parametric)
        stat, p = mannwhitneyu(g0, g1)
        
        # Effect size (Cohen's d)
        effect_size = cohens_d(g0, g1)
        
        results_6m.append({
            'feature': col,
            'p_value': p,
            'mean_readmitted': g0.mean(),
            'mean_dead': g1.mean(),
            'effect_size': effect_size
        })


# =========================================================
# RESULTS RANKING
# =========================================================

df_6m = pd.DataFrame(results_6m).sort_values('p_value')

# Select top features
top_features_6m = df_6m.head(12)


# =========================================================
# VISUALIZATION: BOXPLOTS
# =========================================================

# Extract feature names
top_12_6m = top_features_6m['feature'].tolist()

# Create grid
fig, axes = plt.subplots(4, 3, figsize=(14, 17))

fig.suptitle(
    'Distributions of Top Features: Death Within 6 Months',
    fontsize=16
)

# Plot distributions
for i, feature in enumerate(top_12_6m):

    ax = axes[i // 3, i % 3]
    
    sns.boxplot(
        data=df,
        x='death.within.6.months',
        y=feature,
        ax=ax,
        palette='Set2',
        hue='death.within.6.months',
        legend=False
    )
    
    ax.set_title(feature)
    ax.set_xlabel("Death within 6 Months")
    ax.set_ylabel("Value")


plt.tight_layout()
plt.show()

By looking at the boxplot, we see that the values are pretty different in the two cases.

Hence, we choose that removing 57 observation for the stability of our outcome is the right choice.

In [ ]:
df = df[df['death.within.6.months'] != 1]

In [ ]:
features_to_remove = [
    'death.within.6.months'
]

df = df.drop(columns=features_to_remove)

In [ ]:
print("Number of columsn and patients after filtering:", df.shape)

## Missing Data Analysis

In this section, we analyze the extent of missing values 
across all variables in the dataset.

In [ ]:
# =========================================================
# SUPPRESS WARNINGS (CLEAN OUTPUT)
# =========================================================

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)


# =========================================================
# COMPUTE MISSING VALUES PERCENTAGE
# =========================================================

# Calculate percentage of missing values per feature
missing = (df.isna().mean() * 100).sort_values(ascending=False)


# =========================================================
# SPLIT FEATURES BY MISSINGNESS THRESHOLD
# =========================================================


# Define masks
mask_high = missing > 15
mask_low = missing <= 15

# Apply masks
high_missing = missing[mask_high]
low_missing = missing[mask_low]


# =========================================================
# COLOR MAPPING FUNCTION
# =========================================================

def colors_from_values(values, palette_name):
    """
    Generate color mapping based on normalized values.
    Ensures consistent visual encoding across plots.
    """
    
    # Handle edge case: all values identical
    if values.max() == values.min():
        return np.array(sns.color_palette(palette_name, len(values)))
    
    # Normalize values between 0 and 1
    normalized = (values - values.min()) / (values.max() - values.min())
    
    # Map to palette indices
    indices = np.round(normalized * (len(values) - 1)).astype(int)
    
    palette = sns.color_palette(palette_name, len(values))
    
    return np.array(palette).take(indices, axis=0)


# Generate global color mapping
colors_global = colors_from_values(missing, "RdYlGn_r")

# Split colors consistently with data
colors_high = colors_global[mask_high]
colors_low = colors_global[mask_low]


# =========================================================
# PLOT: HIGH MISSINGNESS FEATURES (> 15%)
# =========================================================

if not high_missing.empty:
    
    # Dynamic figure height based on number of features
    fig_height_high = max(4, len(high_missing) * 0.15)
    
    plt.figure(figsize=(16, fig_height_high))
    
    sns.barplot(
        x=high_missing.values,
        y=high_missing.index,
        orient='h',
        palette=list(colors_high),
        hue=high_missing.index,
        legend=False
    )
    
    # Add reference thresholds
    plt.axvline(x=15, linestyle='--', color='black', alpha=0.5)
    plt.axvline(x=50, linestyle='--', color='black', alpha=0.5)

    # Annotate percentages on bars
    for i, val in enumerate(high_missing.values):
        plt.text(val + 0.5, i, f"{val:.1f}%", va='center')
        
    plt.title("High Missing Values (> 15%)", fontsize=14, fontweight='bold')
    plt.xlabel("Percentage Missing (%)")
    plt.ylabel("Feature")
    
    plt.tight_layout()
    plt.show()

In [ ]:
# =========================================================
# PLOT: LOW MISSINGNESS FEATURES (≤ 15%)
# =========================================================

if not low_missing.empty:
    
    fig_height_low = max(4, len(low_missing) * 0.15)
    
    plt.figure(figsize=(16, fig_height_low))
    
    sns.barplot(
        x=low_missing.values,
        y=low_missing.index,
        orient='h',
        palette=list(colors_low),
        hue=low_missing.index,
        legend=False
    )
    
    for i, val in enumerate(low_missing.values):
        plt.text(val + 0.1, i, f"{val:.1f}%", va='center')
        
    plt.title("Low Missing Values (≤ 15%)", fontsize=14, fontweight='bold')
    plt.xlabel("Percentage Missing (%)")
    plt.ylabel("Feature")
    
    plt.tight_layout()
    plt.show()

## Feature Variability Analysis

We evaluate the **relative variability** of numerical features 
using the **Coefficient of Variation**.

It is defined as:

- Standard Deviation / Mean

It provides a normalized measure of dispersion, allowing comparison 
across variables with different scales.

In [ ]:
# =========================================================
# COEFFICIENT OF VARIATION ANALYSIS
# =========================================================

# Select only numerical features
numeric_cols = df.select_dtypes(include=[np.number])

# Compute Coefficient of Variation:
# CV = standard deviation / absolute mean
# Absolute mean is used to avoid issues with negative values
cv = (
    numeric_cols.std() / numeric_cols.mean().abs()
).sort_values(ascending=True)

# Select top 20 features with lowest Coefficient of Variation
lowest_20_cv = cv.head(20)


# =========================================================
# VISUALIZATION
# =========================================================

plt.figure(figsize=(12, 8))

sns.barplot(
    x=lowest_20_cv.values,
    y=lowest_20_cv.index,
    palette='viridis',
    hue=lowest_20_cv.index,
    legend=False
)

for i, val in enumerate(lowest_20_cv.values):
    plt.text(
        val + (lowest_20_cv.max() * 0.01),
        i,
        f"{val:.4g}",
        va='center'
    )

plt.title(
    "Top 20 Features with the Lowest Relative Variance (Coefficient of Variation)",
    fontsize=16,
    fontweight='bold'
)

plt.xlabel("Coefficient of Variation (Lower = Less relative variation)")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

We find that:
- body temperature blood gas
- pH
- body temperature
- sodium
- sodium ion

have really low variance, let's analyze them in detail.

In [ ]:
# =========================================================
# DEFINE LOW VARIANCE FEATURES
# =========================================================

low_var_features = [
    'body.temperature.blood.gas',
    'pH',
    'body.temperature',
    'sodium',
    'sodium.ion'
]

# =========================================================
# PLOT 1: HISTOGRAMS + KDE (DISTRIBUTIONS)
# =========================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

fig.suptitle(
    'Distributions of Low Variance Features',
    fontsize=16,
    fontweight='bold'
)

axes = axes.flatten()

for i, feature in enumerate(low_var_features):
    
    clean_data = df[feature].dropna()
    
    sns.histplot(
        clean_data,
        ax=axes[i],
        kde=True,
        bins=30,
        color='teal'
    )
    
    axes[i].set_title(
        f"{feature}\n(Variance: {clean_data.var():.4f})"
    )
    
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Count")


fig.delaxes(axes[5])

plt.tight_layout()
plt.show()


# =========================================================
# PLOT 2: BOXPLOTS (OUTLIERS AND CENTRAL TENDENCY)
# =========================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

fig.suptitle(
    'Boxplots of Low Variance Features',
    fontsize=16,
    fontweight='bold'
)

axes = axes.flatten()

for i, feature in enumerate(low_var_features):
    
    clean_data = df[feature].dropna()
    
    sns.boxplot(
        y=clean_data,
        ax=axes[i],
        color='lightcoral'
    )
    
    axes[i].set_title(feature)
    axes[i].set_ylabel("Value")


fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

Some clinical variables (e.g., sodium, pH, body temperature) are naturally 
tightly regulated within physiological ranges.

Clearly the most problematic features that we can instantly remove are 'body.temperature.blood.gas' (0 variance) and 'cholinesterase' (all NA's).

In [ ]:
features_to_remove = [
    'cholinesterase',
    'body.temperature.blood.gas'
]

df = df.drop(columns=features_to_remove)

## Variability Analysis of boolean Features

Unlike numerical features, variability is defined as:

- **1 − frequency of the most common category**

In [ ]:
# =========================================================
# SELECT CATEGORICAL / BOOLEAN VARIABLES
# =========================================================

bool_cols = df.select_dtypes(
    include=['object', 'bool']
)

# =========================================================
# COMPUTE CATEGORICAL VARIABILITY
# =========================================================

cat_variability = {}

for col in bool_cols.columns:

    # Compute relative frequency of most common category
    top_freq = (
        bool_cols[col]
        .value_counts(normalize=True, dropna=False)
        .iloc[0]
    )

    # Variability metric:
    # 1 - proportion of dominant category
    variability = 1 - top_freq

    cat_variability[col] = variability


cat_variability = (
    pd.Series(cat_variability)
    .sort_values(ascending=True)
)

lowest_20_cat = cat_variability.head(20)

# =========================================================
# VISUALIZATION
# =========================================================

plt.figure(figsize=(12, 8))

sns.barplot(
    x=lowest_20_cat.values,
    y=lowest_20_cat.index,
    palette='magma',
    hue=lowest_20_cat.index,
    legend=False
)

for i, val in enumerate(lowest_20_cat.values):

    plt.text(
        val + (lowest_20_cat.max() * 0.01),
        i,
        f"{val:.4f}",
        va='center'
    )

plt.title(
    "Top 20 Boolean Features with Lowest Variability",
    fontsize=16,
    fontweight='bold'
)

plt.xlabel(
    "Categorical Variability (1 - Most Common Category Frequency)"
)

plt.ylabel("Feature")

plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# IDENTIFY LOW VARIABILITY BOOLEAN
# =========================================================

results = []

for col in bool_cols:

    counts = df[col].value_counts(dropna=False)
    
    freqs = (
        df[col]
        .value_counts(normalize=True, dropna=False) * 100
    )

    categories = counts.index.tolist()

    cat1 = categories[0]
    count1 = counts.iloc[0]
    freq1 = freqs.iloc[0]

    if len(categories) > 1:
        cat2 = categories[1]
        count2 = counts.iloc[1]
        freq2 = freqs.iloc[1]
    else:
        cat2 = "NONE"
        count2 = 0
        freq2 = 0


    variability = 100 - freq1

    if variability < 1:

        results.append([
            col,
            cat1,
            count1,
            round(freq1, 3),
            cat2,
            count2,
            round(freq2, 3),
            round(variability, 3)
        ])


# =========================================================
# CREATE SUMMARY TABLE
# =========================================================

summary_df = pd.DataFrame(
    results,
    columns=[
        'Feature',
        'Category_1',
        'Count_1',
        'Freq_1_%',
        'Category_2',
        'Count_2',
        'Freq_2_%',
        'Variability_%'
    ]
)


# =========================================================
# SORT AND DISPLAY
# =========================================================

summary_df = summary_df.sort_values('Variability_%')

print(summary_df.to_string(index=False))

Based on the previous variability analysis, we remove categorical features 
that are nearly constant across the dataset.

These variables were identified as having:

- Variability < 1%

In [ ]:
# =========================================================
# REMOVE NEAR-CONSTANT FEATURES
# =========================================================

low_variability_features = summary_df['Feature'].tolist()

print("Features removed:")
print(low_variability_features)

df = df.drop(
    columns=low_variability_features,
    errors='ignore'
)

print("\nUpdated dataset shape:")
print(df.shape)

## Clinical Data Validation Using Physiological Ranges

To ensure data quality, we validate key clinical variables against 
predefined physiological ranges derived from medical knowledge.

Traditional statistical methods (e.g., z-score, IQR) may fail in clinical data because
extreme values may still be physiologically valid

In [ ]:
clinical_vars = [

    'body.temperature',
    'pulse',
    'respiration',
    'systolic.blood.pressure',
    'diastolic.blood.pressure',
    'map',

    'weight',
    'height',
    'BMI',

    'oxygen.saturation',
    'pH',
    'fio2',

    'glucose.blood.gas',
    'lactate'
]

In [ ]:
# =========================================================
# CLINICAL VALID RANGES (DOMAIN-DRIVEN THRESHOLDS)
# =========================================================

valid_ranges = {

    # -----------------------------------------------------
    # VITAL SIGNS
    # -----------------------------------------------------
    
    'body.temperature': (30, 43),      # Celsius: extreme hypothermia to hyperpyrexia
    'pulse': (20, 220),                # Heart rate (beats per minute)
    'respiration': (5, 60),            # Respiratory rate (breaths per minute)

    'systolic.blood.pressure': (50, 300),   # mmHg
    'diastolic.blood.pressure': (30, 200),  # mmHg
    'map': (40, 200),                       # Mean arterial pressure

    # -----------------------------------------------------
    # ANTHROPOMETRIC VARIABLES
    # -----------------------------------------------------
    
    'weight': (20, 300),              # kg
    'height': (1.0, 2.5),             # meters
    'BMI': (10, 70),                  # Body Mass Index

    # -----------------------------------------------------
    # OXYGENATION / BLOOD GAS
    # -----------------------------------------------------
    
    'oxygen.saturation': (50, 100),   # %
    'pH': (6.8, 7.8),                 # Blood pH
    'fio2': (21, 100),                # Fraction of inspired oxygen (%)

    # -----------------------------------------------------
    # METABOLIC VARIABLES
    # -----------------------------------------------------
    
    'glucose.blood.gas': (1, 40),     # mmol/L
    'lactate': (0.1, 20)              # mmol/L
}

In [ ]:
def get_implausible_values(df, col, low, high):

    subset = df[[col]].copy()

    # -----------------------------------------------------
    # DETECT VALUES OUTSIDE VALID RANGE
    # -----------------------------------------------------
    
    mask = (subset[col] < low) | (subset[col] > high)

    # Subset of implausible values
    extreme = subset[mask].copy()

    if extreme.empty:
        return extreme

    median = df[col].median()

    # Store original value explicitly
    extreme['value'] = extreme[col]

    # Distance from median → severity of anomaly
    extreme['distance_from_median'] = (
        extreme[col] - median
    ).abs()

    return extreme.sort_values(
        'distance_from_median',
        ascending=False
    )

In [ ]:
# =========================================================
# SUMMARY OF IMPLAUSIBLE VALUES ACROSS CLINICAL VARIABLES
# =========================================================

summary_list = []

# Loop through all clinical variables
for col in clinical_vars:

    # Skip missing columns
    if col not in df.columns:
        continue

    # Get clinically valid range
    low, high = valid_ranges[col]

    # Detect implausible values
    extreme = get_implausible_values(df, col, low, high)

    # Store summary only if anomalies exist
    if not extreme.empty:
        summary_list.append(
            pd.DataFrame(
                {
                    'feature': col,
                    'n_implausible': len(extreme),
                    'min': extreme[col].min(),
                    'max': extreme[col].max()
                },
                index=[0]
            )
        )


# =========================================================
# CREATE SUMMARY TABLE
# =========================================================

summary_df = pd.concat(summary_list, ignore_index=True)

summary_df = summary_df.sort_values(
    'n_implausible',
    ascending=False
)

print(summary_df.to_string(index=False))

In [ ]:
# =========================================================
# DETAILED INSPECTION OF TOP FEATURES
# =========================================================

features_to_inspect = summary_df['feature']

for col in features_to_inspect:

    low, high = valid_ranges[col]

    extreme = get_implausible_values(df, col, low, high)

    print("\n" + "=" * 60)
    print(f"FEATURE: {col}  | valid range: [{low}, {high}]")
    print("=" * 60)

    print(
        extreme[['value', 'distance_from_median']]
        .head(15)
        .to_string()
    )

In [ ]:
# ===============================
# BMI
# ===============================
df.loc[[319, 561, 316, 334, 378, 387, 465, 536], 'BMI'] = np.nan

# ===============================
# WEIGHT
# ===============================
df.loc[[378, 387, 465, 536], 'weight'] = np.nan

# ===============================
# HEIGHT
# ===============================
df.loc[[319, 561, 316, 334], 'height'] = np.nan

# ===============================
# MAP
# ===============================
df.loc[[606], 'map'] = np.nan

# ===============================
# BLOOD PRESSURE
# ===============================
df.loc[[606], 'systolic.blood.pressure'] = np.nan
df.loc[[606], 'diastolic.blood.pressure'] = np.nan

## Missing Data Analysis

We analyze the proportion of missing values across all dataset features.

Missingness is computed as percentage of missing observations per variable.

In [ ]:
# 1. Prepare data and split it
missing = (df.isna().mean() * 100).sort_values(ascending=False)

mask_high = missing > 15
mask_low = missing <= 15

high_missing = missing[mask_high]
low_missing = missing[mask_low]

# =========================================================
# PLOT 1: High NA Features (> 15%)
# =========================================================
if not high_missing.empty:
    # Calculate figure height dynamically based on the number of features (approx 0.15 inches per feature)
    fig_height_high = max(4, len(high_missing) * 0.15)
    plt.figure(figsize=(16, fig_height_high))
    
    sns.barplot(
        x=high_missing.values,
        y=high_missing.index,
        orient='h',
        palette=list(colors_high),
        hue=high_missing.index,
        legend=False
    )
    
    # Add vertical reference lines
    plt.axvline(x=15, linestyle='--', color='black', alpha=0.5)
    plt.axvline(x=50, linestyle='--', color='black', alpha=0.5)

    # Add text labels
    for i, val in enumerate(high_missing.values):
        plt.text(val + 0.5, i, f"{val:.1f}%", va='center')
        
    plt.title("High Missing Values (> 15%)", fontsize=14, fontweight='bold')
    plt.xlabel("Percentage Missing (%)")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

## Missingness as Clinical Signal (MNAR Assumption)

In clinical datasets, missing values are often not random (MNAR).

In particular, the absence of certain tests may carry clinical meaning:

- Advanced diagnostics are only requested for patients in critical conditions
- Healthy or stable patients typically do not undergo these examinations

Therefore, missingness itself can be a strong proxy for clinical severity.

Instead of imputing these highly missing variables, we convert them into binary indicators:

- 1 → test was performed (patient clinically indicated)
- 0 → test was not performed (no clinical indication)

In [ ]:
# =========================================================
# MNAR-BASED FEATURE ENGINEERING (CLINICAL FLAGS)
# =========================================================

new_flags = {}

# =========================================================
# A. Advanced Echo Flag
# =========================================================

echo_cols = [
    'LVEF',
    'mitral.valve.EMS',
    'mitral.valve.AMS',
    'EA',
    'tricuspid.valve.return.velocity',
    'tricuspid.valve.return.pressure'
]

avail_echo = [c for c in echo_cols if c in df.columns]

if avail_echo:
    new_flags['Had_Advanced_Echo'] = df[avail_echo].notna().any(axis=1).astype(bool)


# =========================================================
# B. ABG Flag
# =========================================================

abg_cols = [
    'pH',
    'partial.pressure.of.carbon.dioxide',
    'partial.oxygen.pressure',
    'oxygen.saturation',
    'anion.gap'
]

avail_abg = [c for c in abg_cols if c in df.columns]

if avail_abg:
    new_flags['Had_ABG_Drawn'] = df[avail_abg].notna().any(axis=1).astype(bool)


# =========================================================
# C. Extended Electrolytes
# =========================================================

Extended_Electrolytes = [
    'Inorganic.Phosphorus',
    'serum.magnesium'
]

avail_electr = [c for c in Extended_Electrolytes if c in df.columns]

if avail_electr:
    new_flags['Had_Extended_Electrolytes'] = df[avail_electr].notna().any(axis=1).astype(bool)


# =========================================================
# D. Advanced Lipid/Risk Panel
# =========================================================

Advanced_Lipid = [
    'apolipoprotein.A',
    'apolipoprotein.B',
    'lipoprotein',
    'homocysteine'
]

avail_lipid = [c for c in Advanced_Lipid if c in df.columns]

if avail_lipid:
    new_flags['Had_Advanced_Lipid'] = df[avail_lipid].notna().any(axis=1).astype(bool)


# =========================================================
# E. Inflammation Panel Flag
# =========================================================

inflammation_cols = [
    'high.sensitivity.protein',
    'erythrocyte.sedimentation.rate'
]

avail_inflam = [c for c in inflammation_cols if c in df.columns]

if avail_inflam:
    new_flags['Had_Inflammation_Panel'] = df[avail_inflam].notna().any(axis=1).astype(bool)


# =========================================================
# MERGE INTO DATAFRAME
# =========================================================

df = df.copy()

df = pd.concat([df, pd.DataFrame(new_flags, index=df.index)], axis=1)

flags = [
    'Had_Advanced_Echo',
    'Had_ABG_Drawn',
    'Had_Extended_Electrolytes',
    'Had_Advanced_Lipid',
    'Had_Inflammation_Panel'
]

target_col = 're.admission.within.6.months'

print("=== CHI-SQUARE TEST RESULTS ===\n")

for flag in flags:

    if flag in df.columns:

        contingency_table = pd.crosstab(df[flag], df[target_col])

        chi2, p_value, dof, expected = chi2_contingency(contingency_table)

        print(f"--- {flag} ---")
        print(contingency_table)
        print(f"p-value: {p_value:.5f}")

        if p_value < 0.05:
            print("SIGNIFICANT: association detected.\n")
        else:
            print("NOT SIGNIFICANT: no association detected.\n")


# =========================================================
# PLOT OUTCOMES BASED ON FLAGS
# =========================================================

flags = ['Had_Advanced_Echo', 'Had_ABG_Drawn', 'Had_Extended_Electrolytes', 'Had_Advanced_Lipid', 'Had_Inflammation_Panel']
target_col = 're.admission.within.6.months'

# Define a nice color palette for the outcomes
palette_colors = sns.color_palette("Set2")

for flag in flags:
    # Check if the flag was successfully created
    if flag in df.columns:
        plt.figure(figsize=(8, 6))
        
        # Create a grouped bar chart
        ax = sns.countplot(
            data=df, 
            x=flag, 
            hue=target_col, 
            palette=palette_colors
        )
        
        # Add titles and labels
        flag_name_clean = flag.replace('_', ' ')
        plt.title(f"Hospitalization Outcome by {flag_name_clean}", fontsize=14, fontweight='bold')
        plt.xlabel(f"{flag_name_clean} (0 = No, 1 = Yes)", fontsize=12)
        plt.ylabel("Number of Patients", fontsize=12)
        
        # Add value counts on top of the bars for clarity
        for p in ax.patches:
            height = p.get_height()
            # Handle NaN heights (when a category has 0 count)
            if height > 0:
                ax.annotate(f'{int(height)}', 
                            (p.get_x() + p.get_width() / 2., height), 
                            ha='center', va='bottom', 
                            fontsize=10, color='black', xytext=(0, 3), 
                            textcoords='offset points')
                
        plt.legend(title="Outcome", bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()

Had_Extended_Electrolytes does not influence statistically the outcome, we can discard the columns due to high NA's.

Had_Advanced_Lipid does not influence statistically the outcome, we can discard the columns due to high NA's.

In [ ]:
df = df.drop(
    columns=Extended_Electrolytes,
    errors='ignore'
)

In [ ]:
df = df.drop(
    columns=Advanced_Lipid,
    errors='ignore'
)

## Arterial Blood Gas (ABG) Panel Missingness Structure

We analyze the internal structure of missingness within the Arterial Blood Gas (ABG) panel.

Instead of treating each feature independently, we consider the panel as a grouped clinical test.

In [ ]:
# Define the features that belong to the Arterial Blood Gas (ABG) panel
abg_panel_features = [
    'pH', 'standard.residual.base', 'standard.bicarbonate', 
    'partial.pressure.of.carbon.dioxide', 'total.carbon.dioxide', 
    'methemoglobin', 'hematocrit.blood.gas', 'reduced.hemoglobin', 
    'glucose.blood.gas', 'lactate', 'measured.residual.base', 
    'measured.bicarbonate', 'carboxyhemoglobin', 'body.temperature.blood.gas', 
    'oxygen.saturation', 'partial.oxygen.pressure', 'oxyhemoglobin', 'anion.gap',
    'total.hemoglobin', 'sodium.ion', 'chloride.ion', 'free.calcium', 'potassium.ion'
]

# Ensure we only check columns that actually exist in your dataset
avail_abg_features = [col for col in abg_panel_features if col in df.columns]
total_abg_features = len(avail_abg_features)

# Count how many of these specific ABG features are missing (NaN) for each patient
df['missing_abg_count'] = df[avail_abg_features].isna().sum(axis=1)

# Create the plot
plt.figure(figsize=(12, 7))

# Plot the distribution using a countplot
ax = sns.countplot(
    data=df, 
    x='missing_abg_count', 
    color='steelblue',
    edgecolor='black'
)

# Add value counts on top of each bar
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}', 
                    (p.get_x() + p.get_width() / 2., height), 
                    ha='center', va='bottom', 
                    fontsize=10, color='black', xytext=(0, 5), 
                    textcoords='offset points')

plt.title(f"The 'Package Deal' Problem:\nNumber of Missing ABG Features per Patient (Total: {total_abg_features})", 
          fontsize=14, fontweight='bold')
plt.xlabel("Number of Missing Features from the ABG Panel", fontsize=12)
plt.ylabel("Number of Patients", fontsize=12)

# Highlight the extremes
plt.axvline(x=0, color='green', linestyle='--', alpha=0.5, label='All tests present')
plt.axvline(x=2, color='red', linestyle='--', alpha=0.5, label='All tests missing')

plt.legend()
plt.tight_layout()
plt.show()

# Drop the temporary column used for plotting
df = df.drop(columns=['missing_abg_count'])

All ABG variables originate from a single arterial blood gas analysis performed in a single laboratory run.

As a consequence:

- Either the entire panel is present
- Or the entire panel is missing

There is no meaningful intermediate state.

In [ ]:
df = df.drop(
    columns=abg_panel_features,
    errors='ignore'
)

We evaluate whether the variable `respiratory.support.` provides unique information 
or if it is redundant given existing oxygenation-related features:

- `oxygen.inhalation`
- `fio2`

In [ ]:
# =========================================================
# REDUNDANCY CHECK: RESPIRATORY SUPPORT VS OXYGENATION
# =========================================================

cols_to_check = [
    'respiratory.support.',
    'oxygen.inhalation',
    'fio2'
]

# Keep only columns that exist in dataset
cols_to_check = [c for c in cols_to_check if c in df.columns]

# Show first rows for visual comparison
print("\n=== FIRST ROWS COMPARISON ===\n")
print(df[cols_to_check].head(10))


In [ ]:
target_col = 'respiratory.support.'
df = df.drop(columns=[target_col])

In [ ]:
print("Number of columsn and patients after filtering:", df.shape)

We visualize features with substantial missingness (>15%) 
to highlight variables with potential data quality issues or structural absence.

In [ ]:
# 1. Prepare data and split it
missing = (df.isna().mean() * 100).sort_values(ascending=False)

mask_high = missing > 15
mask_low = missing <= 15

high_missing = missing[mask_high]
low_missing = missing[mask_low]

# =========================================================
# PLOT 1: High NA Features (> 15%)
# =========================================================
if not high_missing.empty:
    # Calculate figure height dynamically based on the number of features (approx 0.15 inches per feature)
    fig_height_high = max(4, len(high_missing) * 0.15)
    plt.figure(figsize=(16, fig_height_high))
    
    sns.barplot(
        x=high_missing.values,
        y=high_missing.index,
        orient='h',
        palette=list(colors_high),
        hue=high_missing.index,
        legend=False
    )
    
    # Add vertical reference lines
    plt.axvline(x=15, linestyle='--', color='black', alpha=0.5)
    plt.axvline(x=50, linestyle='--', color='black', alpha=0.5)

    # Add text labels
    for i, val in enumerate(high_missing.values):
        plt.text(val + 0.5, i, f"{val:.1f}%", va='center')
        
    plt.title("High Missing Values (> 15%)", fontsize=14, fontweight='bold')
    plt.xlabel("Percentage Missing (%)")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

We analyze the missingness pattern within the echocardiography (Echo) panel.

In [ ]:
# 1. Define the Echo features
echo_features = [
    'left.ventricular.end.diastolic.diameter.LV', 
    'mitral.valve.EMS', 
    'tricuspid.valve.return.velocity', 
    'LVEF', 
    'mitral.valve.AMS', 
    'EA', 
    'tricuspid.valve.return.pressure'
]


# Ensure columns exist in dataset
avail_features = [c for c in echo_features if c in df.columns]
total_features = len(avail_features)

if total_features > 0:
    # Calculate how many features from this specific panel are missing per patient
    missing_counts = df[avail_features].isna().sum(axis=1)
    
    # 2. Create a single figure (no subplots)
    plt.figure(figsize=(8, 6))
    
    # 3. Plot (added hue and legend=False to suppress Seaborn warnings)
    ax = sns.countplot(
        x=missing_counts, 
        hue=missing_counts, 
        palette='viridis', 
        edgecolor='black',
        legend=False
    )
    
    plt.title(f"Echocardiogram (Echo)\n(Total Features: {total_features})", fontsize=14, fontweight='bold')
    plt.xlabel("Number of Missing Features", fontsize=12)
    plt.ylabel("Number of Patients", fontsize=12)
    
    # Add the numerical labels on top of the bars
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            plt.annotate(f'{int(height)}', 
                         (p.get_x() + p.get_width() / 2., height), 
                         ha='center', va='bottom', 
                         fontsize=11, color='black', xytext=(0, 5), 
                         textcoords='offset points')

    plt.tight_layout()
    plt.show()

In [ ]:
# Ensure we only use columns that exist in the dataset
avail_echo = [c for c in echo_features if c in df.columns]

# 2. Create the "Echo Completeness Score"
# This counts how many NON-MISSING values the patient has for the Echo panel (0 to 7)
df['Echo_Completeness_Score'] = df[avail_echo].notna().sum(axis=1)

# 3. Analyze the relationship between the Score and the Outcome
target_col = 're.admission.within.6.months' # Change to death.within.6.months if preferred

# Calculate the readmission rate for each score level
echo_analysis = df.groupby('Echo_Completeness_Score')[target_col].mean() * 100
echo_counts = df.groupby('Echo_Completeness_Score').size()

# Combine into a dataframe for easy viewing
echo_summary = pd.DataFrame({
    'Total_Patients': echo_counts,
    'Readmission_Rate_Percentage': echo_analysis
}).reset_index()

print("=== Echo Completeness vs. Readmission Rate ===")
print(echo_summary.round(2).to_string(index=False))

# 4. Plot the results
plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=echo_summary, 
    x='Echo_Completeness_Score', 
    y='Readmission_Rate_Percentage',
    palette='magma'
)

# Add percentage labels on top of the bars
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height:.1f}%', 
                    (p.get_x() + p.get_width() / 2., height), 
                    ha='center', va='bottom', 
                    fontsize=11, color='black', xytext=(0, 3), 
                    textcoords='offset points')

plt.title("Readmission Rate by Echo Completeness Score\n(Higher score = More detailed ultrasound performed)", fontsize=14, fontweight='bold')
plt.xlabel("Number of Echo Features Successfully Measured (0 to 7)", fontsize=12)
plt.ylabel("Readmission Rate (%)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
df['Echo_Completeness_Score'] = (
    df['Echo_Completeness_Score']
    .astype('category')
)

df['Echo_Completeness_Score'] = (
    df['Echo_Completeness_Score']
    .cat.as_ordered()
)

Based on the observed panel structure, we simplify the echocardiography representation.

We retain:

- `left.ventricular.end.diastolic.diameter.LV`
- A derived indicator of exam intensity (count of performed measurements)

All other correlated Echo variables are removed.

In [ ]:
echo_features_drop = [
    'mitral.valve.EMS', 
    'tricuspid.valve.return.velocity', 
    'LVEF', 
    'mitral.valve.AMS', 
    'EA', 
    'tricuspid.valve.return.pressure',
    'Had_Advanced_Echo'
]

df = df.drop(
    columns=echo_features_drop,
    errors='ignore'
)

In [ ]:
print("Number of columsn and patients after filtering:", df.shape)

In [ ]:
# 1. Prepare data and split it
missing = (df.isna().mean() * 100).sort_values(ascending=False)

mask_high = missing > 15
mask_low = missing <= 15

high_missing = missing[mask_high]
low_missing = missing[mask_low]

# =========================================================
# PLOT 1: High NA Features (> 15%)
# =========================================================
if not high_missing.empty:
    # Calculate figure height dynamically based on the number of features (approx 0.15 inches per feature)
    fig_height_high = max(4, len(high_missing) * 0.15)
    plt.figure(figsize=(16, fig_height_high))
    
    sns.barplot(
        x=high_missing.values,
        y=high_missing.index,
        orient='h',
        palette=list(colors_high),
        hue=high_missing.index,
        legend=False
    )
    
    # Add vertical reference lines
    plt.axvline(x=15, linestyle='--', color='black', alpha=0.5)
    plt.axvline(x=50, linestyle='--', color='black', alpha=0.5)

    # Add text labels
    for i, val in enumerate(high_missing.values):
        plt.text(val + 0.5, i, f"{val:.1f}%", va='center')
        
    plt.title("High Missing Values (> 15%)", fontsize=14, fontweight='bold')
    plt.xlabel("Percentage Missing (%)")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

Remark on the feature that still have lots of NA's:

- 'myoglobin': A protein released when muscle tissue is damaged, we already have high.sensitivity.troponin and creatine.kinase (CK) in the dataset, which are more specific and widely used biomarkers for myocardial injury in contemporary clinical practice. Myoglobin is less specific and largely considered an early but non-specific marker compared to troponin-based diagnosis. 

- 'high.sensitivity.protein', 'erythrocyte.sedimentation.rate': These variables may reflect clinical suspicion of systemic inflammation. However, similar information is already partially captured by more routinely available hematological markers such as white blood cell count, neutrophil count, and lymphocyte count, which are more complete in the dataset.

- 'glutamic.oxaliplatin': This variable appears to be part of a liver-related biochemical panel, although its exact clinical definition is unclear from the dataset schema. It is likely redundant given the presence of a comprehensive liver function test panel including AST, ALT, ALP, albumin, and bilirubin fractions.

- 'total.bile.acid': The dataset already contains a comprehensive liver function panel including AST, ALT, ALP, GGT, and bilirubin fractions. These markers collectively provide strong coverage of hepatocellular injury and cholestatic patterns, which may reduce the incremental value of total bile acid measurements.

- 'fucosidase': This is a highly specialized biomarker typically ordered in suspected oncological or severe hepatic conditions. The dataset already includes broader clinical indicators such as solid tumor and liver disease flags, which may provide more stable and generalizable predictive information for mortality or readmission models.

- 'nucleotidase': 5'-nucleotidase and GGT are both associated with cholestatic liver injury and may provide partially overlapping information. However, GGT is more commonly available in routine clinical panels, making it more consistently observed in the dataset.


In [ ]:
cols_to_drop = [
    'myoglobin', 
    'high.sensitivity.protein', 
    'glutamic.oxaliplatin', 
    'erythrocyte.sedimentation.rate',
    'total.bile.acid',
    'fucosidase',
    'nucleotidase'
]

# Drop them from the df
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])


After clinical feature reduction and redundancy filtering, the dataset is largely complete.

The only remaining variable with substantial missingness is:

- `left.ventricular.end.diastolic.diameter.LV` (34.4% missing)

In [ ]:
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# 1. Prepare data (Assuming 'df' is your dataframe)
# Filter out columns with 0% missing to keep the plot clean
missing = (df.isna().mean() * 100)
missing = missing[missing > 0].sort_values(ascending=False)

# 2. Calculate colors globally to maintain consistent color mapping
def colors_from_values(values, palette_name):
    # Avoid division by zero if all values are identical
    if values.max() == values.min():
        return np.array(sns.color_palette(palette_name, len(values)))
    normalized = (values - values.min()) / (values.max() - values.min())
    indices = np.round(normalized * (len(values) - 1)).astype(int)
    palette = sns.color_palette(palette_name, len(values))
    return np.array(palette).take(indices, axis=0)

# Generate the color array for the exact length of our missing series
colors_global = colors_from_values(missing, "RdYlGn_r")

# =========================================================
# PLOT: All Missing Features
# =========================================================
if not missing.empty:
    # Calculate figure height dynamically based on the number of features (approx 0.15 inches per feature)
    fig_height = max(4, len(missing) * 0.15)
    plt.figure(figsize=(16, fig_height))
    
    sns.barplot(
        x=missing.values,
        y=missing.index,
        orient='h',
        palette=list(colors_global), # Pass the global colors directly here
        hue=missing.index,
        legend=False
    )
    
    # Add vertical reference lines
    plt.axvline(x=15, linestyle='--', color='black', alpha=0.5)
    plt.axvline(x=50, linestyle='--', color='black', alpha=0.5)

    # Add text labels
    for i, val in enumerate(missing.values):
        plt.text(val + 0.5, i, f"{val:.1f}%", va='center')
        
    plt.title("Percentage of Missing Values per Feature", fontsize=16, fontweight='bold')
    plt.xlabel("Percentage Missing (%)")
    plt.ylabel("Feature")
    
    # Expand x-axis slightly so the text labels don't get cut off
    plt.xlim(0, missing.max() + 5) 
    
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found in the dataset!")

In [ ]:
print("Number of columns and patients after filtering:", df.shape)

## Medication Data Integration

We transform raw medication logs into aggregated counts of clinical treatment categories per patient, merging these new features into the main dataset

In [ ]:
def merge_medication_data(main_df, med_df):
    """
    Cleans the medication dataset, counts drugs by category, 
    and merges those counts into the main clinical dataset.
    """
    X_transformed = main_df.copy()
    md = med_df.copy()
    
    drug_categories = {
        'Count_Diuretics': [
            'FUROSEMIDE INJECTION', 'FUROSEMIDE TABLET', 'TORASEMIDE TABLET', 
            'SPIRONOLACTONE TABLET', 'HYDROCHLOROTHIAZIDE TABLET'
        ],
        'Count_Beta_Blockers': [
            'METOPROLOL SUCCINATE SUSTAINED-RELEASE TABLET', 'METOPROLOL TARTRATE INJECTION'
        ],
        'Count_RAAS_Inhibitors': [
            'BENAZEPRIL HYDROCHLORIDE TABLET', 'VALSARTAN DISPERSIBLE TABLET'
        ],
        'Count_Inotropes': [
            'DIGOXIN TABLET', 'DESLANOSIDE INJECTION', 'DOBUTAMINE HYDROCHLORIDE INJECTION', 
            'ISOPRENALINE HYDROCHLORIDE INJECTION', 'MILRINONE INJECTION'
        ],
        'Count_Vasodilators': [
            'NITROGLYCERIN INJECTION', 'ISOSORBIDE MONONITRATE SUSTAINED RELEASE TABLET'
        ],
        'Count_Antithrombotics': [
            'ASPIRIN ENTERIC-COATED TABLET', 'CLOPIDOGREL HYDROGEN SULPHATE TABLET', 
            'WARFARIN SODIUM TABLET', 'HEPARIN SODIUM INJECTION', 'ENOXAPARIN SODIUM INJECTION'
        ],
        'Count_Lipid_Lowering': [
            'ATORVASTATIN CALCIUM TABLET'
        ],
        'Count_Metabolic_Adjuvants': [
            'MEGLUMINE ADENOSINE CYCLOPHOSPHATE FOR INJECTION', 'SULFOTANSHINONE SODIUM INJECTION', 
            'SHENFU INJECTION'
        ]
    }
    
    # 1. Clean drug names and add taken flag
    if 'Drug_name' in md.columns:
        md['Drug_name'] = md['Drug_name'].astype(str).str.strip().str.upper()
    md['taken'] = 1
    
    # 2. Pivot to wide format
    md_wide = md.pivot_table(
        index='inpatient.number', 
        columns='Drug_name', 
        values='taken', 
        aggfunc='max', 
        fill_value=0
    ).reset_index()
    
    # 3. Create category count columns
    for category_name, drugs_in_category in drug_categories.items():
        available_drugs = [d for d in drugs_in_category if d in md_wide.columns]
        if available_drugs:
            md_wide[category_name] = md_wide[available_drugs].sum(axis=1)
        else:
            md_wide[category_name] = 0
            
    # 4. Filter to keep only inpatient.number and Count_ columns
    cols_to_keep = ['inpatient.number'] + list(drug_categories.keys())
    cols_to_keep = [col for col in cols_to_keep if col in md_wide.columns]
    md_wide = md_wide[cols_to_keep]
    
    # 5. Merge the baseline clinical data with the new medication data
    X_merged = pd.merge(X_transformed, md_wide, on='inpatient.number', how='left')
    
    # 6. Fill patients with NO medication records with 0s ONLY in the new medication columns
    count_columns = list(drug_categories.keys())
    X_merged[count_columns] = X_merged[count_columns].fillna(0)
    
    # 7. Drop 'inpatient.number' as it is a unique identifier and shouldn't be used for ML training
    if 'inpatient.number' in X_merged.columns:
        X_merged = X_merged.drop(columns=['inpatient.number'])
        
    return X_merged

In [ ]:
merged_df = merge_medication_data(df, md_df)

In [ ]:
print("Number of columns and patients after filtering:", df.shape)
print("Number of columns and patients in medication:", md_df.shape)
print("Number of columns and patients after merging and reshaping:", merged_df.shape)

## Data Splitting and Imputation Strategy

The dataset is split into training and test sets using a stratified approach to preserve the prevalence of the outcome variable.

Missing continuous variables are imputed using Multiple Imputation by Chained Equations (MICE), implemented with a Random Forest estimator to capture non-linear relationships between clinical variables.
To ensure physiological plausibility:

- negative imputed values are disallowed
- missingness indicators are retained to preserve informative missing patterns

In [ ]:
# =========================================================
# 1. DEFINE FEATURES (X) AND TARGET (y)
# =========================================================
# We assume 're.admission.within.6.months' is our primary target.
target_col = 're.admission.within.6.months'

cols_to_drop = [
    target_col
]

# Ensure we only drop columns that actually exist in the dataframe
X = merged_df.drop(columns=[col for col in cols_to_drop if col in merged_df.columns])
y = merged_df[target_col].astype(int) # Convert boolean True/False to 1/0

# =========================================================
# 2. TRAIN / TEST SPLIT (80% Train, 20% Test)
# =========================================================
# stratify=y ensures both sets have the exact same percentage of readmissions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} patients")
print(f"Test set size: {X_test.shape[0]} patients")

# =========================================================
# 3. SET UP MICE IMPUTATION
# =========================================================
# MICE operates mathematically, so we separate numerical columns from text/categorical ones
numerical_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# Initialize the Iterative Imputer
mice_imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=20, random_state=42, n_jobs=-1),
    max_iter=5,
    random_state=42,
    min_value=0,        # Clinical safety check: No negative biological values
    add_indicator=True  # Creates a binary flag noting which exact values were imputed
)

# =========================================================
# 4. FIT ON TRAIN, TRANSFORM ON BOTH
# =========================================================
print("\nTraining MICE on X_train... ")
# FIT computes the relationships. TRANSFORM fills the blanks.
X_train_imputed_array = mice_imputer.fit_transform(X_train[numerical_cols])

print("Applying trained MICE to X_test... ")
# TRANSFORM ONLY. We do NOT fit on the test data.
X_test_imputed_array = mice_imputer.transform(X_test[numerical_cols])

# =========================================================
# 5. RECONSTRUCT THE DATAFRAMES
# =========================================================
# Because add_indicator=True, MICE outputs MORE columns than it took in. 
# We fetch the new dynamic column names here:
new_num_cols = mice_imputer.get_feature_names_out(numerical_cols)

X_train_imputed = pd.DataFrame(X_train_imputed_array, columns=new_num_cols, index=X_train.index)
X_test_imputed = pd.DataFrame(X_test_imputed_array, columns=new_num_cols, index=X_test.index)

# Merge any non-numerical text columns back in (if you have them like gender, occupation)
non_num_cols = [c for c in X.columns if c not in numerical_cols]
if non_num_cols:
    X_train_final = pd.concat([X_train[non_num_cols], X_train_imputed], axis=1)
    X_test_final = pd.concat([X_test[non_num_cols], X_test_imputed], axis=1)
else:
    X_train_final = X_train_imputed
    X_test_final = X_test_imputed

print("\nImputation Complete!")
print(f"Final missing values in X_train: {X_train_final.isna().sum().sum()}")
print(f"Final missing values in X_test: {X_test_final.isna().sum().sum()}")

## Correlation Structure Analysis

We compute the correlation matrix of all numerical features after imputation to assess potential multicollinearity and redundancy in the final feature space.

In [ ]:
# =========================================================
# CORRELATION ANALYSIS
# =========================================================

numeric_df = (
    X_train_final
    .select_dtypes(include=np.number)
)

numeric_df = numeric_df[
    [
        c for c in numeric_df.columns
        if not c.startswith('missingindicator')
    ]
]

# =========================================================
# COMPUTE CORRELATION MATRIX
# =========================================================

corr = numeric_df.corr().abs()


In [ ]:
plt.figure(figsize=(16,12))

sns.heatmap(
    corr,
    cmap='coolwarm',
    center=0,
    linewidths = 0.2
)

plt.title("Correlation Matrix")
plt.show()

To identify redundancy in the feature space, we extract pairs of variables with high absolute correlation (> 0.7).

In [ ]:
upper = corr.where(
    np.triu(
        np.ones(corr.shape),
        k=1
    ).astype(bool)
)

high_corr = (
    upper
    .stack()
    .reset_index()
)

high_corr.columns = [
    'feature_1',
    'feature_2',
    'correlation'
]

high_corr = (
    high_corr[
        high_corr['correlation'] > 0.7
    ]
    .sort_values(
        'correlation',
        ascending=False
    )
)

print("\nTop Correlated Feature Pairs (>|0.7|)\n")
print(
    high_corr.head(30).to_string(index=False)
)

The following selection criteria were applied:

- INR was retained instead of prothrombin time ratio due to its standardization and comparability across laboratories.

- Hemoglobin was prioritized as a primary marker of anemia severity, while hematocrit and red blood cell count were considered partially redundant but potentially complementary.

- White blood cell count and neutrophil count were both retained, as they capture different aspects of immune response (global inflammation vs. acute bacterial response).

- Mean corpuscular volume (MCV) was retained as a representative red blood cell morphology index.

- Platelet count was retained as the primary thrombocyte measure, while derived indices were excluded due to redundancy.

- Total bilirubin was retained as a global marker of hepatic function, while direct and indirect bilirubin were removed due to higher granularity but partial redundancy.

- Systolic and diastolic blood pressure were retained, while mean arterial pressure (MAP) was removed as a derived variable.

- Eosinophil absolute count was retained instead of eosinophil ratio to avoid compositional dependency.

- Lactate dehydrogenase (LDH) was retained instead of hydroxybutyrate dehydrogenase due to broader clinical availability.

- LDL cholesterol was prioritized as a more clinically relevant cardiovascular risk marker compared to total cholesterol.

- AST and ALT were both retained due to their complementary roles in hepatocellular injury assessment.

In [ ]:
# =========================================================
# REMOVE MULTICOLLINEAR FEATURES
# (AFTER MICE IMPUTATION)
# =========================================================

cols_to_drop = [

    # Coagulation
    'prothrombin.time.ratio',

    # Red blood cell related
    'hematocrit',
    'red.blood.cell',

    # Erythrocyte morphology
    'mean.hemoglobin.volume',

    # Platelet related
    'platelet.hematocrit',

    # Bilirubin related
    'direct.bilirubin',
    'indirect.bilirubin',

    # Derived blood pressure metric
    'map',

    # Differential count ratios
    'eosinophil.ratio',
    'basophil.ratio',

    # Enzyme redundancy
    'hydroxybutyrate.dehydrogenase',

    # Lipid redundancy
    'cholesterol',

    # Protein-derived ratio
    'white.globulin.ratio',

    # Less specific protein measure
    'total.protein',
]

# Remove from TRAIN
X_train_final = X_train_final.drop(
    columns=cols_to_drop,
    errors='ignore'
)

# Remove from TEST
X_test_final = X_test_final.drop(
    columns=cols_to_drop,
    errors='ignore'
)

print("Updated TRAIN shape:", X_train_final.shape)
print("Updated TEST shape:", X_test_final.shape)

Stessa cosa per categoriche

In [ ]:
# tieni solo categoriche binarie (0/1)
categorical_df = X_train_final[
    [
        c
        for c in X_train_final.columns
        if (
            X_train_final[c]
            .dropna()
            .isin([0,1])
            .all()
        )
    ]
]

# togli missing indicators
categorical_df = categorical_df[
    [
        c
        for c in categorical_df.columns
        if not c.startswith(
            'missingindicator'
        )
    ]
]

# correlazioni
corr = categorical_df.corr().abs()

upper = corr.where(
    np.triu(
        np.ones(corr.shape),
        k=1
    ).astype(bool)
)

high_corr = (
    upper
    .stack()
    .reset_index()
)

high_corr.columns = [
    'feature_1',
    'feature_2',
    'correlation'
]

high_corr = (
    high_corr[
        high_corr['correlation'] > 0.1
    ]
    .sort_values(
        'correlation',
        ascending=False
    )
)

print(
    high_corr.head(20).to_string(index=False)
)

We decided to leave all the boolean features, since the pearson correlation index (in absolute values) is not higher than 0.5.

A final integrity check was performed to ensure that no missing values remain in the training and test sets after preprocessing and imputation.

In [ ]:
# =========================================================
# IDENTIFY REMAINING MISSING VALUES
# =========================================================

# Check the training set
missing_train = X_train_final.isna().sum()
cols_with_na_train = missing_train[missing_train > 0]

print("--- Columns with Missing Values in X_train_final ---")
if not cols_with_na_train.empty:
    print(cols_with_na_train.to_string())
else:
    print("None! The dataset is completely full.")

# Check the test set
missing_test = X_test_final.isna().sum()
cols_with_na_test = missing_test[missing_test > 0]

print("\n--- Columns with Missing Values in X_test_final ---")
if not cols_with_na_test.empty:
    print(cols_with_na_test.to_string())
else:
    print("None! The dataset is completely full.")

In [ ]:
# List of categorical columns to fill
text_columns_to_fill = ['occupation'] 

for col in text_columns_to_fill:
    if col in X_train_final.columns:
        
        # 1. Check if the column is explicitly a 'category' type
        if X_train_final[col].dtype.name == 'category':
            # Add 'Unknown' to the allowed categories for both train and test
            if 'Unknown' not in X_train_final[col].cat.categories:
                X_train_final[col] = X_train_final[col].cat.add_categories('Unknown')
            if 'Unknown' not in X_test_final[col].cat.categories:
                X_test_final[col] = X_test_final[col].cat.add_categories('Unknown')
        
        # 2. Now it is safe to fill the NaNs!
        X_train_final[col] = X_train_final[col].fillna('Unknown')
        X_test_final[col] = X_test_final[col].fillna('Unknown')

# Run a final check to confirm the dataset is 100% full
print(f"Final missing values in X_train: {X_train_final.isna().sum().sum()}")
print(f"Final missing values in X_test: {X_test_final.isna().sum().sum()}")

## Clinical Feature Engineering Strategy

The goal of this feature engineering layer is not to increase dimensionality, but to replace raw redundancy with clinically meaningful composite signals. 

Selected interaction terms are used to model non-linear disease interactions that are clinically well-established but not explicitly captured in raw variables. Examples include:

Structural cardiac dilation combined with neurohormonal stress
Hemodynamic instability indexed by heart rate and blood pressure imbalance
Multi-system inflammatory burden

In [ ]:
class ClinicalFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        # NYHA mapping dictionary
        self.nyha_mapping = {'I': 1, 'II': 2, 'III': 3, 'IV': 4}

    def fit(self, X, y=None):
        return self # Nothing to fit

    def transform(self, X):
        X_transformed = X.copy()
        
        # 1. Transform NYHA to numeric safely
        if 'NYHA.cardiac.function.classification' in X_transformed.columns:
            nyha_raw = X_transformed['NYHA.cardiac.function.classification'].astype(str)
            X_transformed['NYHA_numeric'] = nyha_raw.map(self.nyha_mapping)            
            X_transformed['NYHA_numeric'] = pd.to_numeric(X_transformed['NYHA_numeric'], errors='coerce')

        # 2. Ratio & Difference Engineering
        if 'urea' in X_transformed.columns and 'creatinine.enzymatic.method' in X_transformed.columns:
            X_transformed['BUN_to_Creat_Ratio'] = X_transformed['urea'] / X_transformed['creatinine.enzymatic.method']

        if 'neutrophil.count' in X_transformed.columns and 'lymphocyte.count' in X_transformed.columns:
            X_transformed['NLR'] = X_transformed['neutrophil.count'] / X_transformed['lymphocyte.count']

        if 'platelet' in X_transformed.columns and 'lymphocyte.count' in X_transformed.columns:
            X_transformed['PLR'] = X_transformed['platelet'] / X_transformed['lymphocyte.count']

        if 'systolic.blood.pressure' in X_transformed.columns and 'diastolic.blood.pressure' in X_transformed.columns:
            X_transformed['Pulse_Pressure'] = X_transformed['systolic.blood.pressure'] - X_transformed['diastolic.blood.pressure']

        if 'hemoglobin' in X_transformed.columns and 'NYHA_numeric' in X_transformed.columns:
            X_transformed['Anemic_Severe_HF'] = ((X_transformed['hemoglobin'] < 120) & (X_transformed['NYHA_numeric'] >= 3)).astype(int)

        # ADAPTED: Replaced LVEF_to_BNP with LVEDD_x_BNP_Index
        # A dilated heart (high LVEDD) combined with high stress (high BNP) is a severe risk marker.
        if 'left.ventricular.end.diastolic.diameter.LV' in X_transformed.columns and 'brain.natriuretic.peptide' in X_transformed.columns:
            X_transformed['LVEDD_x_BNP_Index'] = X_transformed['left.ventricular.end.diastolic.diameter.LV'] * X_transformed['brain.natriuretic.peptide']

        if 'creatinine.enzymatic.method' in X_transformed.columns and 'glomerular.filtration.rate' in X_transformed.columns:
            X_transformed['creatinine_to_GFR_Ratio'] = X_transformed['creatinine.enzymatic.method'] / X_transformed['glomerular.filtration.rate']

        if 'pulse' in X_transformed.columns and 'systolic.blood.pressure' in X_transformed.columns:
            X_transformed['Shock_Index'] = X_transformed['pulse'] / (X_transformed['systolic.blood.pressure'] + 1e-5)
       
        if all(col in X_transformed.columns for col in ['sodium', 'chloride', 'carbon.dioxide.binding.capacity']):
            X_transformed['Anion_Gap'] = X_transformed['sodium'] - (X_transformed['chloride'] + X_transformed['carbon.dioxide.binding.capacity'])

        if 'Count_Beta_Blockers' in X_transformed.columns and 'Count_RAAS_Inhibitors' in X_transformed.columns:
            # A simple flag if they are on the two most critical foundational drugs
            X_transformed['On_Basic_GDMT'] = ((X_transformed['Count_Beta_Blockers'] > 0) & 
                                      (X_transformed['Count_RAAS_Inhibitors'] > 0)).astype(int)
          
        if 'cystatin' in X_transformed.columns and 'glomerular.filtration.rate' in X_transformed.columns:
            # Adding a small epsilon to the denominator to prevent division by zero 
            # in the rare event of a zero or erroneously imputed GFR.
            X_transformed['Cystatin_to_GFR_Ratio'] = X_transformed['cystatin'] / (X_transformed['glomerular.filtration.rate'] + 1e-5)

        return X_transformed

## Feature Selection Strategy (ElasticNet + Boruta Hybrid Approach)

To reduce dimensionality while preserving clinically meaningful signal, we apply a hybrid feature selection strategy combining regularized regression and tree-based importance ranking. This dual approach balances linear sparsity assumptions with non-linear interaction sensitivity.

In [ ]:
class ElasticNetFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, cv=5, threshold=1e-4, l1_ratio=0.1):
        self.cv = cv
        self.threshold = threshold
        self.l1_ratio = l1_ratio # 0.5 balances Lasso (sparsity) and Ridge (grouping)
        self.selected_features_ = []

    def fit(self, X, y):
        self.numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        X_numeric = X[self.numeric_cols]

        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(X_numeric)

        # 'saga' solver is required for elasticnet penalty
        self.model = LogisticRegressionCV(
            penalty='elasticnet',
            solver='saga',
            l1_ratios=[self.l1_ratio],
            cv=self.cv,
            random_state=42,
            max_iter=1000, # Increased max_iter because saga can take longer to converge
            scoring='roc_auc',
            n_jobs=-1
        )

        self.model.fit(X_scaled, y)

        # Extract features where coefficient magnitude is above threshold
        coef = self.model.coef_[0]
        self.selected_features_ = [
            self.numeric_cols[i]
            for i in range(len(coef))
            if abs(coef[i]) > self.threshold
        ]

        print(f"[ElasticNet] Selected features: {len(self.selected_features_)}")
        return self

    def transform(self, X):
        cols = [c for c in self.selected_features_ if c in X.columns]
        return X[cols]

    def get_selected_features(self):
        return self.selected_features_

We are applying a "relaxed" Boruta strategy. By dropping perc to 90 we are comparing real features to the 90th percentile of shadow features (Obtained by duplicating the entire dataset, then taking the duplicated columns and randomly shuffles their values. These shuffled clones are called "Shadow Features"). By raising alpha to 0.05 we are making the statistical test more lenient. In order to grasp signals that can be weak, ensuring we don't aggressively drop potentially useful biological markers.

In [ ]:
class BorutaFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, max_depth=5, perc=90, alpha=0.05, keep_tentative=True):
        self.max_depth = max_depth
        self.perc = perc                 # Default 100. Lowering to 90 makes it less strict.
        self.alpha = alpha               # Default 0.05. Raising to 0.10 makes it less strict.
        self.keep_tentative = keep_tentative # Grabs features Boruta was "unsure" about
        self.selected_features_ = []

    def fit(self, X, y):
        self.numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        X_numeric = X[self.numeric_cols]

        rf_estimator = RandomForestClassifier(
            n_jobs=-1, 
            class_weight='balanced', 
            max_depth=self.max_depth,
            random_state=42
        )

        self.boruta_selector = BorutaPy(
            rf_estimator, 
            n_estimators='auto', 
            perc=self.perc,              # <-- Applied here
            alpha=self.alpha,            # <-- Applied here
            verbose=0, 
            random_state=42,
            max_iter=100
        )

        self.boruta_selector.fit(X_numeric.values, y.values)

        # Build the final list including confirmed, and optionally, tentative
        self.selected_features_ = []
        for i in range(len(self.numeric_cols)):
            # If it's a confirmed feature
            if self.boruta_selector.support_[i]:
                self.selected_features_.append(self.numeric_cols[i])
            # If it's a tentative feature and we want to keep them
            elif self.keep_tentative and self.boruta_selector.support_weak_[i]:
                self.selected_features_.append(self.numeric_cols[i])

        print(f"[Boruta] Selected features (Relaxed): {len(self.selected_features_)}")
        return self

    def transform(self, X):
        cols = [c for c in self.selected_features_ if c in X.columns]
        return X[cols]

    def get_selected_features(self):
        return self.selected_features_

In [ ]:
class CombinedFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, combine_strategy='union'):
        self.combine_strategy = combine_strategy
        self.elastic_selector = ElasticNetFeatureSelector()
        self.boruta_selector = BorutaFeatureSelector()

    def fit(self, X, y):
        self.elastic_selector.fit(X, y)
        self.boruta_selector.fit(X, y)

        self.elastic_features_ = self.elastic_selector.get_selected_features()
        self.boruta_features_ = self.boruta_selector.get_selected_features()

        set_elastic = set(self.elastic_features_)
        set_boruta = set(self.boruta_features_)

        self.common_covariates_ = sorted(list(set_elastic & set_boruta))
        self.all_selected_covariates_ = sorted(list(set_elastic | set_boruta))

        print(
            f"[FeatureSelector] "
            f"ElasticNet={len(self.elastic_features_)} | "
            f"Boruta={len(self.boruta_features_)} | "
            f"Common={len(self.common_covariates_)}"
        )

        return self

    def transform(self, X):
        if self.combine_strategy == 'intersection':
            features = self.common_covariates_

            if len(features) == 0:
                print("⚠️ Intersection empty → fallback UNION")
                features = self.all_selected_covariates_
        else:
            features = self.all_selected_covariates_

        cols = [c for c in features if c in X.columns]
        return X[cols]

    def get_elastic_covariates(self):
        return self.elastic_features_

    def get_boruta_covariates(self):
        return self.boruta_features_

    def get_common_covariates(self):
        return self.common_covariates_

This section executes the full preprocessing and feature selection workflow,
transforming raw clinical data into a final model-ready feature space.

In [ ]:
# =========================================================
# STEP 1: FEATURE ENGINEERING
# =========================================================
print("1. Applying Clinical Feature Engineering...")
clinical_engineer = ClinicalFeatureEngineer()

X_train_engineered = clinical_engineer.transform(X_train_final)
X_test_engineered = clinical_engineer.transform(X_test_final)

# =========================================================
# STEP 2: ONE-HOT ENCODING
# =========================================================
print("2. One-Hot Encoding...")
X_train_encoded = pd.get_dummies(X_train_engineered, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_engineered, drop_first=True)

X_train_encoded, X_test_encoded = X_train_encoded.align(
    X_test_encoded, join='left', axis=1, fill_value=0
)

# =========================================================
# STEP 3: FEATURE SELECTION
# =========================================================
print("3. Feature Selection...")

feature_selector = CombinedFeatureSelector(
    combine_strategy='union'
)

X_train_selected = feature_selector.fit_transform(X_train_encoded, y_train)
X_test_selected = feature_selector.transform(X_test_encoded)

# =========================================================
# STEP 4: REPORT
# =========================================================
elastic_list = feature_selector.get_elastic_covariates()
boruta_list = feature_selector.get_boruta_covariates()
common_list = feature_selector.get_common_covariates()

print("\n" + "="*50)
print("✅ PIPELINE EXECUTION COMPLETE")
print("="*50)

print(f"Starting features:         {X_train_final.shape[1]}") # Assuming X_train_final was defined earlier in your code
print(f"After Engineering & OHE:   {X_train_encoded.shape[1]}")
print(f"Final Selected Features:   {X_train_selected.shape[1]}")

print("\n--- FEATURE SELECTION BREAKDOWN ---")
print(f"ElasticNet:        {len(elastic_list)}")
print(f"Boruta:            {len(boruta_list)}")
print(f"Consensus:         {len(common_list)}")

print("\n--- FINAL DATASETS ---")
print(f"Train: {X_train_selected.shape}")
print(f"Test:  {X_test_selected.shape}")


In [ ]:
# 1. Create a DataFrame
df_features = pd.DataFrame({
    'ElasticNet Features': pd.Series(elastic_list),
    'Boruta Features': pd.Series(boruta_list),
    'Strong Consensus': pd.Series(common_list)
}).fillna('') 

# 2. Print the summary headers
print("\n" + "="*80)
print(f"{'ELASTIC NET AND BORUTA COVARIATES':^80}")
print("="*80)
print(f"ElasticNet found: {len(elastic_list)} | Boruta found: {len(boruta_list)} | Consensus: {len(common_list)}\n")

# 3. Display the columns
display(df_features)

## Prediction Model

This section presents the final stage of the pipeline, where the model is trained, optimized and evaluated using clinically meaningful metrics.

We start from a core set of consensus features, identified through a combination of ElasticNet and Boruta.


Model performance is evaluated using a clinical-grade metric suite, including:

- AUC (discrimination ability)
- Precision (positive predictive value)
- Recall (sensitivity)
- Specificity (true negative rate)
- F1-score (balance between precision and recall)

Additionally, we provide:

- Confusion matrices for error analysis
- ROC curves for threshold-independent evaluation

In [ ]:
# =========================================================
# METRICS (EXPLORATORY MODEL EVALUATION)
# =========================================================

def compute_specificity(y_true, y_pred):

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-9)


def evaluate_full(y_true, preds, label="SET"):

    # threshold fixed
    threshold = 0.5
    y_hat = (preds >= threshold).astype(int)

    # metrics
    auc = roc_auc_score(y_true, preds)
    f1 = f1_score(y_true, y_hat)
    recall = recall_score(y_true, y_hat)
    precision = precision_score(y_true, y_hat)
    specificity = compute_specificity(y_true, y_hat)

    cm = confusion_matrix(y_true, y_hat)

    print("\n" + "="*80)
    print(f"📊 {label} RESULTS")
    print("="*80)
    print(f"AUC:         {auc:.4f}")
    print(f"Precision:   {precision:.4f}")
    print(f"Recall:      {recall:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"F1-score:    {f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_hat))

    # =========================================================
    # CONFUSION MATRIX PLOT
    # =========================================================
    plt.figure(figsize=(4,3))
    plt.imshow(cm, cmap="Blues")
    plt.title(f"Confusion Matrix - {label}")
    plt.colorbar()

    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.xticks([0,1], ["Pred 0","Pred 1"])
    plt.yticks([0,1], ["True 0","True 1"])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

    # =========================================================
    # ROC CURVE
    # =========================================================
    fpr, tpr, _ = roc_curve(y_true, preds)

    plt.figure(figsize=(5,4))
    plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
    plt.plot([0,1],[0,1],"--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve - {label}")
    plt.legend()
    plt.show()

    return {
        "auc": auc,
        "f1": f1,
        "recall": recall,
        "precision": precision,
        "specificity": specificity
    }


## MODEL 0: BASELINE LOGISTIC REGRESSION WITH ALL THE FEATURES

In [ ]:
# =========================================================
# BASELINE LOGISTIC REGRESSION (ALL FEATURES)
# =========================================================

print("\n" + "="*80)
print("🟢 BASELINE MODEL: LOGISTIC REGRESSION (ALL FEATURES)")
print("="*80)

# =========================================================
# DATA SETUP
# =========================================================

X_train_base = X_train_encoded.copy()
X_test_base  = X_test_encoded.copy()

print(f"Number of features used: {X_train_base.shape[1]}")

# =========================================================
# MODEL DEFINITION
# =========================================================

baseline_model = LogisticRegression(
    max_iter=5000,
    solver="lbfgs",
    class_weight="balanced"
)

# =========================================================
# TRAINING
# =========================================================

baseline_model.fit(X_train_base, y_train)

# =========================================================
# PREDICTIONS
# =========================================================

train_pred_base = baseline_model.predict_proba(X_train_base)[:, 1]
test_pred_base  = baseline_model.predict_proba(X_test_base)[:, 1]


In [ ]:
# =========================================================
# FINAL EVALUATION
# =========================================================

train_metrics = evaluate_full(y_train, train_pred_base, "LOGISTIC TRAIN")
test_metrics  = evaluate_full(y_test, test_pred_base, "LOGISTIC TEST")

# =========================================================
# FINAL OUTPUT
# =========================================================

print("\n" + "="*80)
print("FINAL SELECTED FEATURES")
print("="*80)
print(set(elastic_list).union(set(boruta_list)))

print("\nTOTAL FEATURES SELECTED:", len(set(elastic_list).union(set(boruta_list))))

## MODEL 1: LOGISTIC REGRESSION WITH BATCH FORWARD FEATURE SELECTION

Logistic Regression trained on features selected through ElasticNet and Boruta (intersection).

The feature space is further refined using a batch-wise forward selection process, where groups of variables are iteratively added only if they yield a measurable improvement in cross-validated AUC (Δ ≥ 0.002).

In [ ]:
# =========================================================
# INPUT FEATURES
# =========================================================

base_features = list(common_list)  # 22 consensus features
all_features = set(elastic_list).union(set(boruta_list))
candidate_features = list(all_features - set(base_features))

print("="*80)
print(f"BASE FEATURES (Consensus): {len(base_features)}")
print(f"CANDIDATE FEATURES: {len(candidate_features)}")
print("="*80)

# =========================================================
# DATA SETUP
# =========================================================

X_train_fs = X_train_selected.copy()
X_test_fs  = X_test_selected.copy()

current_features = base_features.copy()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# =========================================================
# CV FUNCTION
# =========================================================

def cv_auc_score(features):

    X = X_train_fs[features].values
    aucs = []

    for tr_idx, val_idx in cv.split(X, y_train):

        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = LogisticRegression(
            max_iter=3000,
            solver="lbfgs",
            class_weight="balanced",
            C=0.7
        )

        model.fit(X_tr, y_tr)
        preds = model.predict_proba(X_val)[:, 1]

        aucs.append(roc_auc_score(y_val, preds))

    return np.mean(aucs)

# =========================================================
# BASE SCORE
# =========================================================

best_score = cv_auc_score(current_features)

print("\n" + "="*80)
print(f"BASE CV AUC: {best_score:.4f}")
print("="*80)

# =========================================================
# BATCH FORWARD SELECTION 
# =========================================================

batch_size = 5
min_gain = 0.002

iteration = 0
improved = True

while improved and len(candidate_features) > 0:

    iteration += 1
    improved = False

    print("\n" + "#"*80)
    print(f"ITERATION {iteration} | Candidates left: {len(candidate_features)}")
    print("#"*80)

    best_candidate_group = None
    best_candidate_score = best_score

    # shuffle for robustness
    np.random.shuffle(candidate_features)

    batches = [
        candidate_features[i:i + batch_size]
        for i in range(0, len(candidate_features), batch_size)
    ]

    for batch in batches:

        trial_features = current_features + batch
        score = cv_auc_score(trial_features)

        print(f"Batch: {batch} | AUC: {score:.4f}")

        if score > best_candidate_score + min_gain:
            best_candidate_score = score
            best_candidate_group = batch

    # accept best batch
    if best_candidate_group is not None:

        current_features += best_candidate_group

        for f in best_candidate_group:
            if f in candidate_features:
                candidate_features.remove(f)

        best_score = best_candidate_score
        improved = True

        print("\n" + "-"*80)
        print(f"ADDED FEATURES: {best_candidate_group}")
        print(f"NEW CV AUC: {best_score:.4f}")
        print(f"TOTAL FEATURES: {len(current_features)}")
        print("-"*80)

    else:
        print("\nNo improvement found → stopping selection")

# =========================================================
# FINAL MODEL
# =========================================================

print("\n" + "="*80)
print("TRAINING FINAL MODEL")
print("="*80)

final_model = LogisticRegression(
    max_iter=5000,
    solver="lbfgs",
    class_weight="balanced",
    C=0.7
)

final_model.fit(
    X_train_fs[current_features],
    y_train
)

# =========================================================
# PREDICTIONS
# =========================================================

train_pred = final_model.predict_proba(X_train_fs[current_features])[:, 1]
test_pred  = final_model.predict_proba(X_test_fs[current_features])[:, 1]

In [ ]:
# =========================================================
# FINAL EVALUATION
# =========================================================

train_metrics = evaluate_full(y_train, train_pred, "LOGISTIC TRAIN")
test_metrics  = evaluate_full(y_test, test_pred, "LOGISTIC TEST")

# =========================================================
# FINAL OUTPUT
# =========================================================

print("\n" + "="*80)
print("FINAL SELECTED FEATURES")
print("="*80)
print(current_features)

print("\nTOTAL FEATURES SELECTED:", len(current_features))

## MODEL 2: XGBoost + Optuna

XGBoost classifier trained on a feature set defined as the union of ElasticNet and Boruta selections.

Hyperparameters are optimized using Optuna with 5-fold stratified cross-validation, exploring a search space strongly regularized (shallow trees, subsampling, pruning) to reduce overfitting and improve generalization. 

The final model incorporates class imbalance handling (scale_pos_weight).

In [ ]:
print("\n" + "="*60)
print("🟢 STEP 4: OPTUNA HYPERPARAMETER TUNING (XGBOOST)")
print("="*60)

# 1. Create the robust feature pool (Union of ElasticNet & Boruta)
final_features = list(set(elastic_list) | set(boruta_list))
print(f"Features in final pool: {len(final_features)}")

X_train_xgb = X_train_encoded[final_features]
X_test_xgb = X_test_encoded[final_features]

# 2. Calculate Class Imbalance Weight dynamically
# This helps the model if your readmission/mortality rate is low (e.g., <20%)
neg_class_count = sum(y_train == 0)
pos_class_count = sum(y_train == 1)
scale_weight = neg_class_count / pos_class_count
print(f"Dynamic scale_pos_weight applied: {scale_weight:.2f}")

# 3. Define the Optuna Objective Function
def objective(trial):
    # Parameter Search Space heavily biased toward Regularization
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 2, 5), # Kept shallow to prevent memorizing noise
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 0.95),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15), # Forces leaf size
        'gamma': trial.suggest_float('gamma', 0, 5), # Prunes branches that don't improve loss
        
        'scale_pos_weight': scale_weight,
        'eval_metric': 'auc',
        'random_state': 42,
        'n_jobs': -1
    }

    model = XGBClassifier(**param)
    
    # 5-Fold Stratified CV to evaluate parameters safely
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Use cross_val_score to get the mean AUC across the 5 folds
    cv_scores = cross_val_score(model, X_train_xgb, y_train, cv=skf, scoring='roc_auc', n_jobs=-1)
    
    return cv_scores.mean()

# 4. Execute the Optuna Study
# Hiding the verbose output of every single trial to keep logs clean
optuna.logging.set_verbosity(optuna.logging.WARNING) 
study = optuna.create_study(direction='maximize')

print("Running 50 Optuna trials. This may take a few minutes...")
study.optimize(objective, n_trials=50)

print(f"\n✅ Optuna Tuning Complete!")
print(f"Best Cross-Validation AUC: {study.best_value:.4f}")
print("Best Parameters Discovered:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")


# 5. Train the final model on the FULL training set using the best parameters
best_params = study.best_params
best_params['scale_pos_weight'] = scale_weight
best_params['eval_metric'] = 'auc'
best_params['random_state'] = 42
best_params['n_jobs'] = -1

final_xgb = XGBClassifier(**best_params)
final_xgb.fit(X_train_xgb, y_train)

# 6. Predict on the completely unseen Test Set
test_preds = final_xgb.predict_proba(X_test_xgb)[:, 1]

In [ ]:
xgb_train_preds = final_xgb.predict_proba(
    X_train_xgb
)[:,1]

xgb_test_preds = final_xgb.predict_proba(
    X_test_xgb
)[:,1]

evaluate_full(
    y_train,
    xgb_train_preds,
    "XGBOOST TRAIN"
)

evaluate_full(
    y_test,
    xgb_test_preds,
    "XGBOOST TEST"
)

## MODEL 3: RBF Support Vector Machine

Support Vector Machine with RBF kernel trained on the union feature set from ElasticNet and Boruta.

The model is embedded in a pipeline with standardization (mandatory for SVM) and optimized via Grid Search with 5-fold stratified cross-validation, tuning the regularization parameter (C) and kernel coefficient (gamma).

Class imbalance is handled through balanced class weights.

In [ ]:
print("\n" + "="*60)
print("🟢 SUPPORT VECTOR MACHINE (RBF KERNEL) PIPELINE")
print("="*60)


X_train_svm = X_train_encoded[final_features]
X_test_svm = X_test_encoded[final_features]

# 2. Build the Pipeline (Scaling is mathematically mandatory for SVM)
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(
        probability=True,  # Mandatory to calculate ROC-AUC
        random_state=42,
        class_weight='balanced',
        cache_size=1000    # Allocates 1GB RAM to speed up kernel math
    ))
])

# 3. Define the Grid Search Space
# C controls the penalty for misclassification. Gamma controls the "reach" of a single patient.
svm_param_grid = {
    'svc__C': [10, 100, 200, 500, 1000],
    'svc__gamma': [0.1, 0.01, 0.001, 0.0001, 0.00001],
    'svc__kernel': ['rbf'] 
}

# 4. Execute Grid Search
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

svm_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_param_grid,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=2
)

print("Running Grid Search for SVM... ")
svm_search.fit(X_train_svm, y_train)

print(f"\n✅ Tuning Complete!")
print("Best Parameters:", svm_search.best_params_)

# 5. Evaluate strictly on Test Set
best_svm = svm_search.best_estimator_


In [ ]:
svm_train_preds = best_svm.predict_proba(
    X_train_svm
)[:,1]

svm_test_preds = best_svm.predict_proba(
    X_test_svm
)[:,1]

evaluate_full(
    y_train,
    svm_train_preds,
    "SVM TRAIN"
)

evaluate_full(
    y_test,
    svm_test_preds,
    "SVM TEST"
)

## MODEL4: LOGISTIC AND SVM -> VOTING SYSTEM

Ensemble model based on soft voting, combining a regularized Logistic Regression and an RBF Support Vector Machine, both trained on the union feature set from ElasticNet and Boruta.

In [ ]:
# 1. Isolate the exact optimized features
print(f"Total unique features in the ensemble pool: {len(final_features)}")

X_train_ens = X_train_encoded[final_features]
X_test_ens = X_test_encoded[final_features]

# =====================================================================
# SUB-MODEL 1: Optimized L2 Logistic Regression 
# =====================================================================
# The SubFeatureSelector is REMOVED. The pipeline directly scales the data.
logreg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(
        max_iter=2000,
        class_weight={0: 1, 1:2} 
    ))
])

# =====================================================================
# SUB-MODEL 2: Optimized RBF Support Vector Machine
# =====================================================================
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(
        C=100,         # Your breakthrough optimal value
        gamma=0.0005,       # Your breakthrough optimal value
        kernel='rbf',
        probability=True,    # Non-negotiable for soft voting
        class_weight='balanced' ,
        random_state=42,
        cache_size=1000
    ))
])

# =====================================================================
# THE ENSEMBLE: Soft Voting Classifier
# =====================================================================
voting_ensemble = VotingClassifier(
    estimators=[
        ('lr_model', logreg_pipeline),
        ('svm_model', svm_pipeline)
    ],
    voting='soft',
    n_jobs=-1
)

print("Training the Voting Ensemble...")
voting_ensemble.fit(X_train_ens, y_train)


In [ ]:
ensemble_train_preds = voting_ensemble.predict_proba(
    X_train_ens
)[:,1]

ensemble_test_preds = voting_ensemble.predict_proba(
    X_test_ens
)[:,1]

evaluate_full(
    y_train,
    ensemble_train_preds,
    "VOTING TRAIN"
)

evaluate_full(
    y_test,
    ensemble_test_preds,
    "VOTING TEST"
)

## MODELLO 5: STACKING ENSEMBLE

Stacking ensemble that combines three heterogeneous base learners: Logistic Regression, XGBoost and Explainable Boosting Machine.

Out-of-fold predictions are generated using 5-fold stratified cross-validation.

A Logistic Regression meta-learner is then trained on these OOF predictions to learn the optimal weighting of each base model

In [ ]:


kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_train_stk = X_train_encoded[final_features]
X_test_stk = X_test_encoded[final_features]

oof_preds_lr = np.zeros(len(X_train_stk))
oof_preds_xgb = np.zeros(len(X_train_stk))
oof_preds_ebm = np.zeros(len(X_train_stk))

test_preds_lr = np.zeros(len(X_test_stk))
test_preds_xgb = np.zeros(len(X_test_stk))
test_preds_ebm = np.zeros(len(X_test_stk))

In [ ]:
# ===============================
# K-FOLD SETUP
# ===============================
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ===============================
# INIT PREDICTIONS
# ===============================
oof_preds_lr = np.zeros(len(X_train_stk))
oof_preds_xgb = np.zeros(len(X_train_stk))
oof_preds_ebm = np.zeros(len(X_train_stk))

test_preds_lr = np.zeros(len(X_test_stk))
test_preds_xgb = np.zeros(len(X_test_stk))
test_preds_ebm = np.zeros(len(X_test_stk))

# ===============================
# CROSS-VALIDATION LOOP
# ===============================
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_stk, y_train)):
    
    print(f"\n--- Fold {fold+1} ---")

    X_tr, X_val = X_train_stk.iloc[train_idx], X_train_stk.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # ===============================
    # MODEL 1: LOGISTIC
    # ===============================
    lr = LogisticRegression(
        max_iter=1000,
        n_jobs=-1
    )
    
    lr.fit(X_tr, y_tr)

    oof_preds_lr[val_idx] = lr.predict_proba(X_val)[:, 1]
    test_preds_lr += lr.predict_proba(X_test_stk)[:, 1] / kf.n_splits


    # ===============================
    # MODEL 2: XGBOOST
    # ===============================
    xgb = XGBClassifier(
        n_estimators=80,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1
    )

    xgb.fit(X_tr.values, y_tr)

    oof_preds_xgb[val_idx] = xgb.predict_proba(X_val.values)[:, 1]
    test_preds_xgb += xgb.predict_proba(X_test_stk.values)[:, 1] / kf.n_splits


    # ===============================
    # MODEL 3: EBM
    # ===============================
    ebm = ExplainableBoostingClassifier(
        random_state=42,
        interactions=0,    
        max_bins=128,     
        n_jobs=-1
    )

    ebm.fit(X_tr, y_tr)

    oof_preds_ebm[val_idx] = ebm.predict_proba(X_val)[:, 1]
    test_preds_ebm += ebm.predict_proba(X_test_stk)[:, 1] / kf.n_splits

In [ ]:
# STACKING
stack_X_train = np.vstack([
    oof_preds_lr,
    oof_preds_xgb,
    oof_preds_ebm
]).T

stack_X_test = np.vstack([
    test_preds_lr,
    test_preds_xgb,
    test_preds_ebm
]).T

meta_model = LogisticRegression()
meta_model.fit(stack_X_train, y_train)

final_preds = meta_model.predict_proba(stack_X_test)[:,1]

In [ ]:
stack_train_preds = meta_model.predict_proba(
    stack_X_train
)[:,1]

stack_test_preds = meta_model.predict_proba(
    stack_X_test
)[:,1]

evaluate_full(
    y_train,
    stack_train_preds,
    "STACKING TRAIN"
)

evaluate_full(
    y_test,
    stack_test_preds,
    "STACKING TEST"
)

### COMPARISON BETWEEN MODEL PERFORMANCES

In [ ]:
# =========================================================
# FINAL MODEL COMPARISON
# ROC CURVES + CONFUSION MATRICES
# =========================================================

models = {
    "Weighted Logistic": test_pred,
    "XGBoost": xgb_test_preds,
    "SVM": svm_test_preds,
    "Voting Ensemble": ensemble_test_preds,
    "Stacking": stack_test_preds
}

# =========================================================
# AUC TABLE
# =========================================================

print("\n" + "="*80)
print("FINAL TEST SET COMPARISON")
print("="*80)

results = []

for name, preds in models.items():

    auc = roc_auc_score(y_test, preds)

    results.append({
        "Model": name,
        "AUC": auc
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("AUC", ascending=False)
    .reset_index(drop=True)
)

print(results_df)

# =========================================================
# ROC CURVES
# =========================================================

plt.figure(figsize=(10,8))

for name, preds in models.items():

    fpr, tpr, _ = roc_curve(y_test, preds)
    auc = roc_auc_score(y_test, preds)

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"{name} (AUC={auc:.3f})"
    )

plt.plot([0,1], [0,1], "k--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison - Test Set")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

# =========================================================
# CONFUSION MATRICES
# =========================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes = axes.flatten()

for idx, (name, preds) in enumerate(models.items()):

    y_pred = (preds >= 0.5).astype(int)

    cm = confusion_matrix(y_test, y_pred)

    ax = axes[idx]

    im = ax.imshow(cm, cmap="Blues")

    ax.set_title(name)

    ax.set_xticks([0,1])
    ax.set_yticks([0,1])

    ax.set_xticklabels(["Pred 0","Pred 1"])
    ax.set_yticklabels(["True 0","True 1"])

    for i in range(2):
        for j in range(2):
            ax.text(
                j,
                i,
                cm[i,j],
                ha="center",
                va="center",
                fontsize=12
            )

# rimuove il sesto subplot vuoto
for k in range(len(models), len(axes)):
    fig.delaxes(axes[k])

plt.tight_layout()
plt.show()

The results show that more complex models (XGBoost, stacking, and ensemble methods) do not outperform the weighted Logistic Regression, which achieves the highest test AUC (0.6851).

In this context, Logistic Regression is the preferred choice because it provides direct clinical interpretability (via odds ratios), strong stability on noisy healthcare data and competitive predictive performance, making it the best balance between accuracy and real-world clinical usability.

### A POSTERIORI ANALYSIS of LOGISTIC MODEL

This code computes the feature importance of the trained logistic regression model by extracting its coefficients and associating them with the corresponding input features.


It then ranks features by their absolute coefficient magnitude and plots the top 15 most influential variables, showing whether each feature increases (positive coefficient) or decreases (negative coefficient) the predicted risk.

In [ ]:



# Extract coefficients and map them to feature names
coefficients = final_model.coef_[0]
feature_importance = pd.DataFrame({
    'Feature': current_features,
    'Coefficient': coefficients,
    'Absolute_Influence': np.abs(coefficients)
})

# Sort by absolute influence to find the most important features
feature_importance = feature_importance.sort_values(by='Absolute_Influence', ascending=False)

# Plot the top 15 most influential features
plt.figure(figsize=(10, 8))
sns.barplot(
    x='Coefficient', 
    y='Feature', 
    data=feature_importance.head(15),
    palette=np.where(feature_importance.head(15)['Coefficient'] > 0, 'crimson', 'steelblue')
)
plt.title("Top 15 Most Influential Features (Log-Odds)")
plt.xlabel("Coefficient Value (Red = Pushes to Class 1, Blue = Pushes to Class 0)")
plt.tight_layout()
plt.show()

This code estimates the uncertainty of the logistic regression coefficients by computing an approximate variance-covariance matrix using the Fisher Information Matrix. It then derives standard errors, confidence intervals and odds ratios for each feature.

Finally, it builds a clinical-style summary table highlighting whether each variable is statistically significant (95% CI excluding 1) and whether it acts as a risk or protective factor.


In [ ]:
# =========================================================
# COMPUTE STANDARD ERRORS (approximation from sklearn)
# =========================================================

X = X_train_fs[current_features].values
y = y_train.values

# Predicted probabilities
p = final_model.predict_proba(X)[:, 1]

# Design matrix with intercept
X_design = np.hstack([np.ones((X.shape[0], 1)), X])

# Weight matrix (diagonal of p*(1-p))
W = np.diag(p * (1 - p))

# Covariance matrix (Fisher Information inverse)
cov_matrix = np.linalg.inv(X_design.T @ W @ X_design)

# Standard errors (exclude intercept)
se = np.sqrt(np.diag(cov_matrix))[1:]

# =========================================================
# EXTRACT COEFFICIENTS
# =========================================================

coef = final_model.coef_[0]

# =========================================================
# CI CALCULATION (CORRECT)
# =========================================================

z = 1.96

lower_coef = coef - z * se
upper_coef = coef + z * se

# Odds ratios
or_val = np.exp(coef)
lower_or = np.exp(lower_coef)
upper_or = np.exp(upper_coef)

# =========================================================
# BUILD TABLE
# =========================================================

df = pd.DataFrame({
    "Feature": current_features,
    "Odds_Ratio": or_val,
    "CI_Lower (OR)": lower_or,
    "CI_Upper (OR)": upper_or
})

# =========================================================
# SIGNIFICANCE
# =========================================================

df["Significant"] = ~((df["CI_Lower (OR)"] <= 1) & (df["CI_Upper (OR)"] >= 1))

df["Effect"] = np.where(
    df["Odds_Ratio"] > 1,
    "↑ Risk",
    "↓ Protective"
)

df["Significance"] = np.where(
    df["Significant"],
    "✔ Significant",
    "✖ Not significant"
)

# =========================================================
# FORMAT STRING
# =========================================================

df["OR_CI"] = df.apply(
    lambda row: f"{row['Odds_Ratio']:.2f} ({row['CI_Lower (OR)']:.2f} – {row['CI_Upper (OR)']:.2f})",
    axis=1
)

# =========================================================
# FINAL TABLE
# =========================================================

final_table = df[[
    "Feature",
    "OR_CI",
    "Effect",
    "Significance"
]]

# sort by importance
final_table = final_table.assign(
    Distance_from_1=np.abs(np.log(df["Odds_Ratio"]))
)
final_table = final_table.sort_values("Distance_from_1", ascending=False)
final_table = final_table.drop(columns=["Distance_from_1"])

# =========================================================
# PRINT
# =========================================================

warnings.filterwarnings("ignore")
significant_table = final_table[
    df["Significant"]
].copy()

print("\n" + "="*80)
print("🟢 SIGNIFICANT FEATURES ONLY (95% CI excludes 1)")
print("="*80)

if len(significant_table) == 0:
    print("⚠️ No statistically significant features found")
else:
    print(significant_table.to_string(index=False))

This code performs a **one-at-a-time marginal effect analysis** by generating synthetic patient profiles where all features are fixed at their mean values except one, which is varied across its observed range. 

The model’s predicted probabilities are then computed to visualize how changes in each individual feature influence the predicted risk of the outcome, holding all other variables constant.

In [ ]:

# 1. Define the list of features you want to analyze
features_to_test = [
'D.dimer', 'NLR', 'PLR', 'Pulse_Pressure', 'albumin', 'basophil.count', 'dischargeDay', 'eosinophil.count', 'globulin', 'glomerular.filtration.rate',
#'high.sensitivity.troponin', 'hydroxybutyrate.dehydrogenase.to.lactate.dehydrogenase', 'left.ventricular.end.diastolic.diameter.LV', 'low.density.lipoprotein.cholesterol',
#'mean.corpuscular.volume', 'missingindicator_left.ventricular.end.diastolic.diameter.LV', 'neutrophil.ratio', 'prothrombin.activity', 'sodium', 
#'standard.deviation.of.red.blood.cell.distribution.width', 'systolic.blood.pressure', 'uric.acid', 'creatinine_to_GFR_Ratio', 'CCI.score', 'missingindicator_D.dimer', 
#'Count_Diuretics', 'missingindicator_white.globulin.ratio', 'body.temperature', 'hemoglobin', 'glutamic.pyruvic.transaminase', 'NYHA_numeric', 'missingindicator_globulin',
#'Count_Metabolic_Adjuvants', 'Cystatin_to_GFR_Ratio', 'missingindicator_standard.deviation.of.red.blood.cell.distribution.width', 'On_Basic_GDMT', 'missingindicator_albumin'

]

# 2. Create the "baseline" instance where every feature is held at its mean (average)
# We calculate this ONCE outside the loop for efficiency
baseline_values = X_train_encoded[current_features].mean().to_dict()

# Loop through each feature to generate its own plot
for feature in features_to_test:
    
    # 3. Create a range of values for the current feature (from its min to max)
    feature_min = X_train_encoded[feature].min()
    feature_max = X_train_encoded[feature].max()
    test_values = np.linspace(feature_min, feature_max, 100)

    # 4. Generate synthetic data keeping everything constant EXCEPT the current feature
    synthetic_data = pd.DataFrame([baseline_values] * 100)
    synthetic_data[feature] = test_values

    # 5. Predict the probabilities for this synthetic data
    predicted_probs = final_model.predict_proba(synthetic_data)[:, 1]

    # 6. Plot the Marginal Effect
    plt.figure(figsize=(8, 5))
    sns.lineplot(x=test_values, y=predicted_probs, color='darkorange', linewidth=3)
    plt.axhline(y=0.5, color='gray', linestyle='--', label='50% Decision Threshold')
    
    # Added a newline character (\n) in the title so long feature names don't get cut off
    plt.title(f"Marginal Effect of\n{feature}", fontsize=14)
    plt.xlabel(f"Values of {feature}")
    plt.ylabel("Predicted Probability of Class 1")
    plt.ylim(0, 1)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

This code applies **SHAP (Shapley Additive Explanations)** to the trained logistic regression model in order to quantify feature contributions to predictions. 


In [ ]:
# 1. Initialize the SHAP Explainer
# We pass the model and the background data (training set) so SHAP knows what the "average" baseline is.
explainer = shap.Explainer(final_model, X_train_encoded[current_features])

# 2. Calculate SHAP values for the test set
# (If your test set is hundreds of thousands of rows, you might want to sample it first: X_test_encoded.sample(1000))
shap_values = explainer(X_test_encoded[current_features])

# =====================================================================
# GLOBAL INTERPRETABILITY: The Beeswarm Plot
# =====================================================================
print("\n" + "="*60)
print("🟢 GLOBAL INTERPRETABILITY (BEESWARM PLOT)")
print("="*60)
# This plot shows the distribution of impacts for all features. 
# Red dots = High feature value, Blue dots = Low feature value.
# Dots shifted to the right pushed the model toward predicting Class 1.
plt.figure()
shap.plots.beeswarm(shap_values)
plt.show()

# =====================================================================
# LOCAL INTERPRETABILITY: The Waterfall Plot
# =====================================================================
print("\n" + "="*60)
print("🟢 LOCAL INTERPRETABILITY (WATERFALL PLOT)")
print("="*60)

# Let's look at the VERY FIRST instance in your test set (index 0).
# You can change this index to investigate specific False Positives or False Negatives.
customer_index = 0 
print(f"Explaining prediction for Test Set Index: {customer_index}")

# The Waterfall plot starts at the bottom with the expected value (E[f(x)], the average prediction).
# Each bar shows how a specific feature pushed the prediction up or down, 
# resulting in the final prediction (f(x)) at the top.
plt.figure()
shap.plots.waterfall(shap_values[customer_index])
plt.show()

This code visualizes how selected clinical features are distributed across different prediction outcomes (true positives, true negatives, false positives, and false negatives) using boxplots. 
By comparing feature distributions between correctly and incorrectly classified cases, it helps identify whether systematic differences in specific variables are associated with model errors.

In [ ]:
# SANITY CHECK: Let's make absolutely sure they are the same length
# (If these print different numbers, you have a bug earlier in your notebook!)
print(f"Rows in X_test_encoded: {len(X_test_encoded)}")
print(f"Rows in y_test:         {len(y_test)}")

# 1. Create the copy
error_analysis_df = X_test_encoded[current_features].copy()

# 2. Reset the index of the new dataframe to be perfectly sequential (0, 1, 2...)
error_analysis_df = error_analysis_df.reset_index(drop=True)

# 3. Add the true labels by resetting y_test's index as well. 
# This ignores the old '1634' index and forces it to match the sequential dataframe.
error_analysis_df['True_Label'] = y_test.reset_index(drop=True)

# 4. Recalculate predictions purely on this aligned data
error_analysis_df['Predicted_Prob'] = final_model.predict_proba(error_analysis_df[current_features])[:, 1]
error_analysis_df['Predicted_Label'] = final_model.predict(error_analysis_df[current_features])

# 5. Define Error Types
conditions = [
    (error_analysis_df['True_Label'] == 1) & (error_analysis_df['Predicted_Label'] == 1), # True Positive
    (error_analysis_df['True_Label'] == 0) & (error_analysis_df['Predicted_Label'] == 0), # True Negative
    (error_analysis_df['True_Label'] == 0) & (error_analysis_df['Predicted_Label'] == 1), # False Positive
    (error_analysis_df['True_Label'] == 1) & (error_analysis_df['Predicted_Label'] == 0)  # False Negative
]
choices = ['True Positive', 'True Negative', 'False Positive', 'False Negative']

error_analysis_df['Error_Type'] = np.select(conditions, choices, default='Unknown')

# --- Profiling the Errors ---
print("\n" + "="*60)
print("🔍 ERROR PROFILING: Mean Feature Values by Prediction Outcome")
print("="*60)

# Group by the error type and calculate the mean for your features
error_profile = error_analysis_df.groupby('Error_Type')[current_features].mean().T
print(error_profile)

In [ ]:


# Let's pick 4 features you suspect might be causing the misclassifications
# (Swap these out with actual column names from your current_features list)
# Replace the previous list with these exact culprits
features_to_plot = [
    'glomerular.filtration.rate', 
    'uric.acid', 
    'missingindicator_left.ventricular.end.diastolic.diameter.LV', 
    'left.ventricular.end.diastolic.diameter.LV'
]

# Set up a 2x2 grid of plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# Define a custom color palette to highlight errors
# Correct = Grays/Blues, Errors = Bright Reds/Oranges
error_palette = {
    'True Positive': '#2ecc71',   # Green
    'True Negative': '#3498db',   # Blue
    'False Positive': '#f39c12',  # Orange
    'False Negative': '#e74c3c'   # Red
}

for i, feature in enumerate(features_to_plot):
    # We use a boxplot to see the mean AND the spread of the data
    sns.boxplot(
        data=error_analysis_df, 
        x='Error_Type', 
        y=feature, 
        ax=axes[i],
        palette=error_palette,
        order=['True Negative', 'True Positive', 'False Positive', 'False Negative']
    )
    
    axes[i].set_title(f'Distribution of {feature} by Outcome')
    axes[i].set_ylabel(feature)
    axes[i].set_xlabel('')
    
    # Rotate x-labels slightly for better readability
    axes[i].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

# FINAL INVESTIGATION

This code generates **subgroup performance analyses** by evaluating model discrimination (AUC) across different patient strata for each variable of interest. 

For every subgroup definition, it plots horizontal bar charts showing AUC values per category, includes a reference line at 0.5 (random performance) and annotates each bar with both AUC and sample size, allowing assessment of model stability, fairness and performance heterogeneity across clinical subpopulations.


In [ ]:
# =========================================================
# FINAL MODEL
# =========================================================

model = final_model

features_used = current_features

X_train_eval = X_train_fs[
    features_used
].copy()

X_test_eval = X_test_fs[
    features_used
].copy()

# =========================================================
# PREDICTIONS
# =========================================================

train_pred = model.predict_proba(
    X_train_eval
)[:,1]

test_pred = model.predict_proba(
    X_test_eval
)[:,1]

# =========================================================
# ORIGINAL TEST DATA
# =========================================================

X_test_raw = X_test_final.copy()

# =========================================================
# SUBGROUP VARIABLES
# =========================================================

subgroup_vars = [

    # Demographics
    "gender",
    "ageCat",
    "occupation",

    # HF characteristics
    "type.of.heart.failure",
    "NYHA.cardiac.function.classification",

    # Comorbidities
    "diabetes",
    "moderate.to.severe.chronic.kidney.disease",
    "Chronic.obstructive.pulmonary.disease",

    # Feature engineering
    "Had_Advanced_Echo",
    "ABG_Completeness",

]
subgroup_vars = [

    v
    for v in subgroup_vars
    if v in X_test_raw.columns

]

# =========================================================
# PRETTY NAMES
# =========================================================

pretty_names = {

    # =====================================================
    # DEMOGRAPHICS
    # =====================================================

    "gender":
        "Gender",

    "ageCat":
        "Age Category",

    "occupation":
        "Occupation",

    # =====================================================
    # HEART FAILURE CHARACTERISTICS
    # =====================================================

    "type.of.heart.failure":
        "Heart Failure Type",

    "NYHA.cardiac.function.classification":
        "NYHA Class",

    # =====================================================
    # COMORBIDITIES
    # =====================================================

    "diabetes":
        "Diabetes",

    "moderate.to.severe.chronic.kidney.disease":
        "Chronic Kidney Disease",

    "Chronic.obstructive.pulmonary.disease":
        "COPD",

    # =====================================================
    # FEATURE ENGINEERING
    # =====================================================

    "Had_Advanced_Echo":
        "Advanced Echocardiography",

    "ABG_Completeness":
        "ABG Completeness"

}
# =========================================================
# OPTIONAL HF LABELS
# =========================================================

hf_mapping = {

    "Left":
        "Left HF",

    "Right":
        "Right HF",

    "Both":
        "Biventricular HF"

}

if "type.of.heart.failure" in X_test_raw.columns:

    X_test_raw[
        "type.of.heart.failure"
    ] = (

        X_test_raw[
            "type.of.heart.failure"
        ]
        .replace(hf_mapping)

    )

# =========================================================
# AUC FUNCTION
# =========================================================

def evaluate_group(
    y_true,
    preds
):

    if len(
        np.unique(y_true)
    ) < 2:

        return np.nan

    return roc_auc_score(
        y_true,
        preds
    )

# =========================================================
# SUBGROUP ANALYSIS
# =========================================================

results = []

for var in subgroup_vars:

    groups = (

        X_test_raw[var]
        .astype("object")
        .fillna("Missing")

    )

    for g in sorted(
        groups.unique()
    ):

        idx = groups == g

        # skip tiny groups
        if idx.sum() < 10:
            continue

        auc = evaluate_group(
            y_test[idx],
            test_pred[idx]
        )

        results.append({

            "Variable":
                pretty_names[var],

            "Group":
                g,

            "AUC":
                auc,

            "N":
                idx.sum()

        })

# =========================================================
# RESULTS TABLE
# =========================================================

res_df = pd.DataFrame(
    results
)

print("\n")
print("="*70)
print("SUBGROUP PERFORMANCE")
print("="*70)

print(
    res_df.round(3)
)

In [ ]:
# =========================================================
# SUBGROUP PLOTS
# =========================================================

for var in res_df[
    "Variable"
].unique():

    df_plot = (

        res_df[
            res_df["Variable"] == var
        ]
        .sort_values(
            "AUC",
            ascending=False
        )

    )

    plt.figure(
        figsize=(8,4)
    )

    plt.barh(

        df_plot["Group"]
        .astype(str),

        df_plot["AUC"]

    )

    plt.axvline(

        0.50,

        linestyle="--",

        color="red",

        label="Random"

    )

    plt.xlim(
        0,
        1
    )

    plt.xlabel(
        "AUC"
    )

    plt.title(
        f"Model Performance by {var}",
        fontweight="bold"
    )

    for i, row in enumerate(
        df_plot.itertuples()
    ):

        plt.text(

            row.AUC + 0.01,

            i,

            f"AUC={row.AUC:.2f}\nN={row.N}",

            va="center",

            fontsize=9

        )

    plt.legend()

    plt.tight_layout()

    plt.show()